# Statistical Analysis of Lifestyle and Health Factors Associated with Cancer Using NHANES Healthcare Data

## Working title:
> Statistical Analysis of Lifestyle and Health Factors Associated with Cancer Using NHANES Healthcare Data

## Problem
> To analyze NHANES healthcare data to identify lifestyle and health-related factors associated with cancer history and develop a data-driven model to classify participants based on reported cancer history.

## Main research question
> How are demographic, lifestyle, dietary, and physical-health factors associated with reported cancer history, and can these factors be used to classify participants according to their reported cancer history?

# About The Dataset

## The project uses NHANES 2013–2014 data. Relevant information is distributed across demographic, questionnaire, examination, and dietary files, which can be linked using SEQN.

## The dataset has 6 different csv files:
- demographic.csv
- diet.csv
- examination.csv
- labs.csv
- medications.csv
- questionnaire.csv


In [1]:
import os
import pandas as pd
import numpy as np

files = [file for file in os.listdir() if file.endswith(".csv")]

print("Dataset files:")
for file in files:
    print(file)

Dataset files:
demographic.csv
diet.csv
examination.csv
final_dataset.csv
labs.csv
medications.csv
processed_dataset.csv
questionnaire.csv


In [2]:
demographic_data = pd.read_csv("demographic.csv")
questionnaire_data = pd.read_csv("questionnaire.csv")
examination_data = pd.read_csv("examination.csv")
diet_data = pd.read_csv("diet.csv")
labs_data = pd.read_csv("labs.csv")

medications_data = pd.read_csv("medications.csv", encoding="latin1") # latin1 encoding is used because the CSV contains non-UTF-8 characters

print("Datasets loaded successfully.")

Datasets loaded successfully.


In [3]:
demographic_data.columns

Index(['SEQN', 'SDDSRVYR', 'RIDSTATR', 'RIAGENDR', 'RIDAGEYR', 'RIDAGEMN',
       'RIDRETH1', 'RIDRETH3', 'RIDEXMON', 'RIDEXAGM', 'DMQMILIZ', 'DMQADFC',
       'DMDBORN4', 'DMDCITZN', 'DMDYRSUS', 'DMDEDUC3', 'DMDEDUC2', 'DMDMARTL',
       'RIDEXPRG', 'SIALANG', 'SIAPROXY', 'SIAINTRP', 'FIALANG', 'FIAPROXY',
       'FIAINTRP', 'MIALANG', 'MIAPROXY', 'MIAINTRP', 'AIALANGA', 'DMDHHSIZ',
       'DMDFMSIZ', 'DMDHHSZA', 'DMDHHSZB', 'DMDHHSZE', 'DMDHRGND', 'DMDHRAGE',
       'DMDHRBR4', 'DMDHREDU', 'DMDHRMAR', 'DMDHSEDU', 'WTINT2YR', 'WTMEC2YR',
       'SDMVPSU', 'SDMVSTRA', 'INDHHIN2', 'INDFMIN2', 'INDFMPIR'],
      dtype='str')

In [4]:
print("Demographic:", demographic_data.shape)
print("Questionnaire:", questionnaire_data.shape)
print("Examination:", examination_data.shape)
print("Diet:", diet_data.shape)
print("Labs:", labs_data.shape)
print("Medications:", medications_data.shape)

Demographic: (10175, 47)
Questionnaire: (10175, 953)
Examination: (9813, 224)
Diet: (9813, 168)
Labs: (9813, 424)
Medications: (20194, 13)


# Observation

The datasets contain different numbers of rows and columns. 
- Demographic and Questionnaire datasets contain 10,175 records
- Examination, Dietary, and Laboratory datasets contain 9,813 records. 
- Medications dataset contains 20,194 records, indicating that it may contain multiple records for some participants. 

Therefore, the datasets cannot be directly combined directly

# Demographic Dataset

The demographic dataset contains information about NHANES participants. For this project, only the demographic and socioeconomic variables relevant to the study of cancer and lifestyle-related factors have been selected. The selected variables include participant ID, gender, age, race/ethnicity, and the ratio of family income to the poverty guidelines.

The selected dataset contains 10,175 participants and 5 variables. 
The `SEQN` variable uniquely identifies each participant and will be used to link the demographic data with the other NHANES datasets.

In [5]:
demographic_data.columns

Index(['SEQN', 'SDDSRVYR', 'RIDSTATR', 'RIAGENDR', 'RIDAGEYR', 'RIDAGEMN',
       'RIDRETH1', 'RIDRETH3', 'RIDEXMON', 'RIDEXAGM', 'DMQMILIZ', 'DMQADFC',
       'DMDBORN4', 'DMDCITZN', 'DMDYRSUS', 'DMDEDUC3', 'DMDEDUC2', 'DMDMARTL',
       'RIDEXPRG', 'SIALANG', 'SIAPROXY', 'SIAINTRP', 'FIALANG', 'FIAPROXY',
       'FIAINTRP', 'MIALANG', 'MIAPROXY', 'MIAINTRP', 'AIALANGA', 'DMDHHSIZ',
       'DMDFMSIZ', 'DMDHHSZA', 'DMDHHSZB', 'DMDHHSZE', 'DMDHRGND', 'DMDHRAGE',
       'DMDHRBR4', 'DMDHREDU', 'DMDHRMAR', 'DMDHSEDU', 'WTINT2YR', 'WTMEC2YR',
       'SDMVPSU', 'SDMVSTRA', 'INDHHIN2', 'INDFMIN2', 'INDFMPIR'],
      dtype='str')

## Columns selected

- SEQN - Respondent sequence number.
- RIAGENDR - gender
- RIDAGEYR - age at the time of screening
- RIDRETH1 - race / ethnicity
- INDFMPIR - A ratio of family income to poverty guidelines.

In [6]:
demographic_selected = demographic_data[
    ["SEQN", "RIAGENDR", "RIDAGEYR", "RIDRETH1", "INDFMPIR"]
]

In [7]:
demographic_selected.dtypes.to_frame("Data Type")

,Data Type
SEQN,int64
RIAGENDR,int64
RIDAGEYR,int64
RIDRETH1,int64
INDFMPIR,float64


In [8]:
demographic_selected.isnull().sum()

SEQN          0
RIAGENDR      0
RIDAGEYR      0
RIDRETH1      0
INDFMPIR    785
dtype: int64

In [9]:
print("Number of duplicate rows:", demographic_selected.duplicated().sum())

Number of duplicate rows: 0


In [10]:
demographic_selected.describe()

,SEQN,RIAGENDR,RIDAGEYR,RIDRETH1,INDFMPIR
count,10175.000000,10175.000000,10175.000000,10175.000000,9390.000000
mean,78644.000000,1.508305,31.484128,3.091892,2.252153
std,2937.413829,0.499956,24.421651,1.263305,1.634907
min,73557.000000,1.000000,0.000000,1.000000,0.000000
25%,76100.500000,1.000000,10.000000,2.000000,0.870000
50%,78644.000000,2.000000,26.000000,3.000000,1.705000
75%,81187.500000,2.000000,52.000000,4.000000,3.607500
max,83731.000000,2.000000,80.000000,5.000000,5.000000


In [11]:
# Check whether each participant has a unique SEQN
print("Total SEQN values:", demographic_selected["SEQN"].count())
print("Unique SEQN values:", demographic_selected["SEQN"].nunique())

Total SEQN values: 10175
Unique SEQN values: 10175


## Observation

- The selected demographic dataset contains 10,175 unique participants and 5 relevant variables:
  1) `SEQN`
  2) `RIAGENDR`
  3) `RIDAGEYR`
  4) `RIDRETH1`
  5) `INDFMPIR`
     
The `SEQN` values are unique for every participant, making them suitable as the common identifier for merging the NHANES datasets.

- The dataset contains information about participants' gender, age, race/ethnicity, and family income-to-poverty ratio.
- The age variable ranges from 0 to 80 years.
- The gender and race/ethnicity variables are stored as coded categorical values and therefore should be interpreted using their corresponding NHANES definitions rather than as numerical measurements.
- No duplicate rows were found in the selected demographic dataset. Most selected variables have complete data, while `INDFMPIR` contains 785 missing values.
   
Overall, the selected demographic variables provide important demographic and socioeconomic information for studying differences in cancer history and lifestyle-related factors across participant groups.

# Questionnaire Dataset

The questionnaire dataset contains self-reported information collected from NHANES participants through health and lifestyle questionnaires. For this project, the dataset will be used to obtain information related to cancer history, smoking, alcohol consumption, and physical activity. These variables will help us study the association between lifestyle factors and reported cancer history.

In [12]:
questionnaire_data.columns

Index(['SEQN', 'ACD011A', 'ACD011B', 'ACD011C', 'ACD040', 'ACD110', 'ALQ101',
       'ALQ110', 'ALQ120Q', 'ALQ120U',
       ...
       'WHD080U', 'WHD080L', 'WHD110', 'WHD120', 'WHD130', 'WHD140', 'WHQ150',
       'WHQ030M', 'WHQ500', 'WHQ520'],
      dtype='str', length=953)

In [13]:
questionnaire_selected = questionnaire_data[
    ["SEQN", "MCQ220",
     "SMQ020", "SMQ040", "SMD650",
     "ALQ101", "ALQ120Q", "ALQ120U", 
     "PAQ650", "PAQ620", "PAQ635"]
]

In [14]:
questionnaire_selected.dtypes.to_frame()

,0
SEQN,int64
MCQ220,float64
SMQ020,float64
SMQ040,float64
SMD650,float64
ALQ101,float64
ALQ120Q,float64
ALQ120U,float64
PAQ650,float64
PAQ620,float64


## Columns Selected

| Column | Description | Use in Project |
|---|---|---|
| `SEQN` | Respondent sequence number | Unique participant ID / merge key |
| `MCQ220` | Ever told by a doctor or health professional that the participant had cancer or a malignancy | Main cancer-history variable |
| `SMQ020` | Ever smoked at least 100 cigarettes in life | Smoking history |
| `SMQ040` | Current smoking status | Current smoking behavior |
| `SMD650` | Average number of cigarettes smoked per day during the past 30 days, on days when the participant smoked | Smoking intensity |
| `ALQ120Q` | Number of days alcohol was consumed per week, month, or year during the past 12 months | Alcohol-use frequency quantity |
| `ALQ120U` | Unit associated with `ALQ120Q`: week, month, or year | Alcohol-frequency unit |
| `ALQ120U` | Unit used to report the frequency | Alcohol-use frequency unit |
| `PAQ650` | Vigorous recreational physical activity | Physical activity |
| `PAQ620` | Moderate work-related physical activity | Physical activity |
| `PAQ635` | Walking or bicycling for transportation | Physical activity |


**Cancer Variables**
- `MCQ220` → Cancer history

**Smoking Variables**
- `SMQ020` → Smoking history
- `SMQ040` → Current smoking status
- `SMD650` → Cigarettes smoked per day

**Alcohol Variables**
- `ALQ101` → Alcohol-use history
- `ALQ120Q` → Alcohol-consumption frequency
- `ALQ120U` → Unit of Alcohol-consumption frequency

**Physical Activity Variables**
- `PAQ650` → Vigorous recreational physical activity
- `PAQ620` → Moderate work-related physical activity
- `PAQ635` → Walking or bicycling for transportation


In [15]:
questionnaire_selected.dtypes.to_frame()

,0
SEQN,int64
MCQ220,float64
SMQ020,float64
SMQ040,float64
SMD650,float64
ALQ101,float64
ALQ120Q,float64
ALQ120U,float64
PAQ650,float64
PAQ620,float64


In [16]:
questionnaire_selected.isnull().sum()

SEQN          0
MCQ220     4406
SMQ020     4062
SMQ040     7596
SMD650     8927
ALQ101     4754
ALQ120Q    5696
ALQ120U    6582
PAQ650     3028
PAQ620     3027
PAQ635     3028
dtype: int64

In [17]:
print(
    "Duplicate questionnaire rows:",
    questionnaire_selected.duplicated().sum()
)

print(
    "Duplicate questionnaire SEQN:",
    questionnaire_selected["SEQN"].duplicated().sum()
)

print(
    "Missing SEQN:",
    questionnaire_selected["SEQN"].isna().sum()
)

Duplicate questionnaire rows: 0
Duplicate questionnaire SEQN: 0
Missing SEQN: 0


In [18]:
# Display the frequency of each response code for the selected questionnaire variables
for column in questionnaire_selected.columns:
    print("\n", column)
    print(questionnaire_selected[column].value_counts(dropna=False))


 SEQN
SEQN
73557    1
73558    1
73559    1
73560    1
73561    1
        ..
83727    1
83728    1
83729    1
83730    1
83731    1
Name: count, Length: 10175, dtype: int64

 MCQ220
MCQ220
2.0    5222
NaN    4406
1.0     547
Name: count, dtype: int64

 SMQ020
SMQ020
NaN    4062
2.0    3532
1.0    2579
9.0       2
Name: count, dtype: int64

 SMQ040
SMQ040
NaN    7596
3.0    1347
1.0     992
2.0     240
Name: count, dtype: int64

 SMD650
SMD650
NaN      8927
10.0      248
20.0      224
5.0       120
1.0       107
3.0        93
2.0        83
15.0       77
4.0        65
6.0        48
7.0        40
30.0       34
12.0       30
8.0        26
40.0       12
9.0         9
25.0        5
11.0        4
13.0        4
17.0        4
18.0        3
999.0       2
50.0        2
14.0        2
16.0        1
60.0        1
27.0        1
90.0        1
29.0        1
28.0        1
Name: count, dtype: int64

 ALQ101
ALQ101
NaN    4754
1.0    3790
2.0    1623
9.0       8
Name: count, dtype: int64

 ALQ120Q
ALQ120

In [19]:
# Display the unique non-missing values for the questionnaire variables
# This helps identify the response codes that need cleaning

for column in questionnaire_selected.columns[1:]:
    print(column, ":", sorted(questionnaire_selected[column].dropna().unique()))

MCQ220 : [np.float64(1.0), np.float64(2.0)]
SMQ020 : [np.float64(1.0), np.float64(2.0), np.float64(9.0)]
SMQ040 : [np.float64(1.0), np.float64(2.0), np.float64(3.0)]
SMD650 : [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(10.0), np.float64(11.0), np.float64(12.0), np.float64(13.0), np.float64(14.0), np.float64(15.0), np.float64(16.0), np.float64(17.0), np.float64(18.0), np.float64(20.0), np.float64(25.0), np.float64(27.0), np.float64(28.0), np.float64(29.0), np.float64(30.0), np.float64(40.0), np.float64(50.0), np.float64(60.0), np.float64(90.0), np.float64(999.0)]
ALQ101 : [np.float64(1.0), np.float64(2.0), np.float64(9.0)]
ALQ120Q : [np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(10.0), np.float64(11.0), np.float64(12.0), np.float64(13.

In [20]:
# Display the special response values that may need cleaning
special_codes = {
    "MCQ220": [7, 9],
    "SMQ020": [7, 9],
    "SMQ040": [7, 9],
    "SMD650": [777, 999],
    "ALQ101": [7, 9],
    "ALQ120Q": [777, 999],
    "ALQ120U": [7, 9],
    "PAQ650": [7, 9],
    "PAQ620": [7, 9],
    "PAQ635": [7, 9]
}

# Identify special response codes
for column, codes in special_codes.items():

    found = questionnaire_selected[
        column
    ].value_counts(dropna=False)

    found = found[found.index.isin(codes)]

    if not found.empty:
        print(f"\n{column}")
        print(found)


SMQ020
SMQ020
9.0    2
Name: count, dtype: int64

SMD650
SMD650
999.0    2
Name: count, dtype: int64

ALQ101
ALQ101
9.0    8
Name: count, dtype: int64

ALQ120Q
ALQ120Q
999.0    4
Name: count, dtype: int64

PAQ650
PAQ650
9.0    1
Name: count, dtype: int64

PAQ620
PAQ620
9.0    2
7.0    1
Name: count, dtype: int64

PAQ635
PAQ635
9.0    1
Name: count, dtype: int64


In [21]:
# Create a copy of the selected questionnaire dataset
questionnaire_cleaned = questionnaire_selected.copy()

# Replace special response codes using variable-specific rules
for column, codes in special_codes.items():

    questionnaire_cleaned[column] = (
        questionnaire_cleaned[column].replace(
            codes, np.nan
        )
    )

print("Variable-specific response codes cleaned successfully.")

Variable-specific response codes cleaned successfully.


In [22]:
# Check missing values after cleaning the special response codes
questionnaire_cleaned.isnull().sum()

SEQN          0
MCQ220     4406
SMQ020     4064
SMQ040     7596
SMD650     8929
ALQ101     4762
ALQ120Q    5700
ALQ120U    6582
PAQ650     3029
PAQ620     3030
PAQ635     3029
dtype: int64

In [23]:
# Verify that variable-specific special codes were removed

for column, codes in special_codes.items():

    remaining = questionnaire_cleaned[
        column
    ].isin(codes).sum()

    print(f"{column}: {remaining} special codes remaining")

MCQ220: 0 special codes remaining
SMQ020: 0 special codes remaining
SMQ040: 0 special codes remaining
SMD650: 0 special codes remaining
ALQ101: 0 special codes remaining
ALQ120Q: 0 special codes remaining
ALQ120U: 0 special codes remaining
PAQ650: 0 special codes remaining
PAQ620: 0 special codes remaining
PAQ635: 0 special codes remaining


In [24]:
# Verify valid response categories

valid_categories = {
    "MCQ220": [1, 2],
    "SMQ020": [1, 2],
    "SMQ040": [1, 2, 3],
    "ALQ101": [1, 2],
    "ALQ120U": [1, 2, 3],
    "PAQ650": [1, 2],
    "PAQ620": [1, 2],
    "PAQ635": [1, 2]
}

for column, valid_values in valid_categories.items():

    unexpected = questionnaire_cleaned.loc[
        questionnaire_cleaned[column].notna() &
        ~questionnaire_cleaned[column].isin(valid_values),
        column
    ].unique()

    print(f"{column}: unexpected values = {unexpected}")

MCQ220: unexpected values = []
SMQ020: unexpected values = []
SMQ040: unexpected values = []
ALQ101: unexpected values = []
ALQ120U: unexpected values = []
PAQ650: unexpected values = []
PAQ620: unexpected values = []
PAQ635: unexpected values = []


In [25]:
print(
    "Invalid SMD650 values:",
    (~questionnaire_cleaned["SMD650"].dropna().between(1, 95)).sum()
)

print(
    "Invalid ALQ120Q values:",
    (~questionnaire_cleaned["ALQ120Q"].dropna().between(0, 365)).sum()
)

Invalid SMD650 values: 0
Invalid ALQ120Q values: 0


In [26]:
# Check consistency between alcohol frequency and frequency unit

alq_check = {
    "Positive frequency with missing unit": (
        (questionnaire_cleaned["ALQ120Q"] > 0) &
        questionnaire_cleaned["ALQ120U"].isna()
    ).sum(),

    "Zero frequency with recorded unit": (
        (questionnaire_cleaned["ALQ120Q"] == 0) &
        questionnaire_cleaned["ALQ120U"].notna()
    ).sum(),

    "Missing frequency with recorded unit": (
        questionnaire_cleaned["ALQ120Q"].isna() &
        questionnaire_cleaned["ALQ120U"].notna()
    ).sum()
}

for description, count in alq_check.items():
    print(f"{description}: {count}")

Positive frequency with missing unit: 0
Zero frequency with recorded unit: 0
Missing frequency with recorded unit: 0


## Observation

- The questionnaire dataset contains 10,175 participant records and 953 original variables.
- For this project, 11 relevant variables were selected:
  1) SEQN - unique participant identifier and contains no missing values, making it suitable for merging with the other NHANES datasets.
  2) MCQ220 - used as the main cancer-history variable. Among its valid responses, 547 participants reported cancer history and 5,222 reported no cancer history.
  3) SMQ020 - Indicates whether a participant has smoked at least 100 cigarettes in their entire life. It helps examine the association between a history of cigarette smoking and reported cancer history.,
  4) SMQ040 - Describes the current smoking frequency of participants who have smoked at least 100 cigarettes in their lifetime, distinguishing daily, some-day, and non-smoking responses. It can help explore differences in cancer history by current smoking status.,
  5) SMD650 - Records the number of cigarettes smoked per day among applicable participants. It may help examine the relationship between cigarette consumption and reported cancer history,
  6) ALQ101 - Indicates whether a participant has had at least 12 alcoholic drinks in any one year. It provides information about past or occasional alcohol consumption history,
  7) ALQ120Q - Records how often a participant drank alcoholic beverages during the period covered by the questionnaire, expressed as a frequency measure. It helps explore patterns of alcohol consumption in relation to cancer history.,
  8) ALQ120U – Indicates the unit in which the alcohol-consumption frequency (ALQ120Q) is reported: per week, per month, or per year. It is essential for correctly interpreting the alcohol-consumption frequency.
  9) PAQ650 - Indicates whether a participant engaged in vigorous recreational activities that cause large increases in breathing or heart rate. It helps examine the relationship between vigorous leisure-time physical activity and cancer history,
  10) PAQ620 - Indicates whether a participant engaged in moderate work-related physical activities that cause small increases in breathing or heart rate. It can be used to explore the association between work-related physical activity and cancer history 
  11) PAQ635 - Indicates whether a participant walks or rides a bicycle for transportation. It provides information about active transportation habits and their association with reported cancer history..

---

- Therefore, among the 5,769 participants with valid cancer-history responses, 9.49% reported a history of cancer.
- The selected variables provide information about cancer history, smoking, alcohol consumption, and physical activity, which are directly relevant to the project's research question.
- Special response codes were identified and handled using variable-specific cleaning rules.
- Categorical variables use 7 and 9 for refused and don't-know responses, while numeric variables such as SMD650 and ALQ120Q use 777 and 999.
- Valid numerical frequency values, including 7 and 9 in ALQ120Q, were retained.
- The cleaned variables were subsequently checked for remaining special codes and invalid values.
- Duplicate participant identifiers and missing SEQN values were also checked; none were found.

In [27]:
questionnaire_cleaned.isnull().sum()

SEQN          0
MCQ220     4406
SMQ020     4064
SMQ040     7596
SMD650     8929
ALQ101     4762
ALQ120Q    5700
ALQ120U    6582
PAQ650     3029
PAQ620     3030
PAQ635     3029
dtype: int64

# Examination Dataset

The examination dataset contains physical examination and measurement information collected from NHANES participants.
It provides objective health-related measurements that can be used to study differences between participants with and without a reported history of cancer. 
For this project, relevant examination variables such as body measurements and other health indicators will be selected to investigate their relationship with cancer history and lifestyle factors. 
The dataset contains 9,813 participant records and 224 variables, and the participant identifier `SEQN` will be used to link the selected examination data with the other NHANES datasets.

In [28]:
examination_data.columns.to_list()

['SEQN',
 'PEASCST1',
 'PEASCTM1',
 'PEASCCT1',
 'BPXCHR',
 'BPAARM',
 'BPACSZ',
 'BPXPLS',
 'BPXPULS',
 'BPXPTY',
 'BPXML1',
 'BPXSY1',
 'BPXDI1',
 'BPAEN1',
 'BPXSY2',
 'BPXDI2',
 'BPAEN2',
 'BPXSY3',
 'BPXDI3',
 'BPAEN3',
 'BPXSY4',
 'BPXDI4',
 'BPAEN4',
 'BMDSTATS',
 'BMXWT',
 'BMIWT',
 'BMXRECUM',
 'BMIRECUM',
 'BMXHEAD',
 'BMIHEAD',
 'BMXHT',
 'BMIHT',
 'BMXBMI',
 'BMDBMIC',
 'BMXLEG',
 'BMILEG',
 'BMXARML',
 'BMIARML',
 'BMXARMC',
 'BMIARMC',
 'BMXWAIST',
 'BMIWAIST',
 'BMXSAD1',
 'BMXSAD2',
 'BMXSAD3',
 'BMXSAD4',
 'BMDAVSAD',
 'BMDSADCM',
 'MGDEXSTS',
 'MGD050',
 'MGD060',
 'MGQ070',
 'MGQ080',
 'MGQ090',
 'MGQ100',
 'MGQ110',
 'MGQ120',
 'MGD130',
 'MGQ90DG',
 'MGDSEAT',
 'MGAPHAND',
 'MGATHAND',
 'MGXH1T1',
 'MGXH1T1E',
 'MGXH2T1',
 'MGXH2T1E',
 'MGXH1T2',
 'MGXH1T2E',
 'MGXH2T2',
 'MGXH2T2E',
 'MGXH1T3',
 'MGXH1T3E',
 'MGXH2T3',
 'MGXH2T3E',
 'MGDCGSZ',
 'OHDEXSTS',
 'OHDDESTS',
 'OHXIMP',
 'OHX01TC',
 'OHX02TC',
 'OHX03TC',
 'OHX04TC',
 'OHX05TC',
 'OHX06TC',
 'OHX07TC',
 

## Columns Selected

**Participant Identifier**
- `SEQN` - Respondent sequence number.

**Body Measurements**
- `BMXBMI` - Body Mass Index (kg/m²).
- `BMXWAIST` - Waist circumference (cm).
- `BMXWT` - Body weight (kg).
- `BMXHT` - Standing height (cm).

**Blood Pressure Measurements**
- `BPXSY1` - Systolic blood pressure, first reading (mm Hg).
- `BPXSY2` - Systolic blood pressure, second reading (mm Hg).
- `BPXSY3` - Systolic blood pressure, third reading (mm Hg).
- `BPXDI1` - Diastolic blood pressure, first reading (mm Hg).
- `BPXDI2` - Diastolic blood pressure, second reading (mm Hg).
- `BPXDI3` - Diastolic blood pressure, third reading (mm Hg).

**Pulse Measurement**
- `BPXPLS` - Pulse rate measured over 60 seconds (beats per minute).

In [29]:
examination_selected = examination_data[
    ["SEQN", "BMXBMI", "BMXWAIST", "BMXWT", "BMXHT",
     "BPXSY1", "BPXSY2", "BPXSY3", "BPXSY4",
     "BPXDI1", "BPXDI2", "BPXDI3", "BPXDI4",
     "BPXPLS"]
].copy()

In [30]:
examination_selected.dtypes

SEQN          int64
BMXBMI      float64
BMXWAIST    float64
BMXWT       float64
BMXHT       float64
BPXSY1      float64
BPXSY2      float64
BPXSY3      float64
BPXSY4      float64
BPXDI1      float64
BPXDI2      float64
BPXDI3      float64
BPXDI4      float64
BPXPLS      float64
dtype: object

In [31]:
examination_selected.duplicated().sum()

np.int64(0)

In [32]:
examination_selected.isnull().sum()

SEQN           0
BMXBMI       758
BMXWAIST    1152
BMXWT         90
BMXHT        746
BPXSY1      2641
BPXSY2      2404
BPXSY3      2405
BPXSY4      9298
BPXDI1      2641
BPXDI2      2404
BPXDI3      2405
BPXDI4      9298
BPXPLS      2264
dtype: int64

In [33]:
# Check missing values in the selected examination dataset

missing_summary = pd.DataFrame({
    "Missing_Count": examination_selected.isna().sum(),
    "Missing_Percentage": (
        examination_selected.isna().mean() * 100
    )
})

display(missing_summary.round(2))

,Missing_Count,Missing_Percentage
SEQN,0,0.00
BMXBMI,758,7.72
BMXWAIST,1152,11.74
BMXWT,90,0.92
BMXHT,746,7.60
BPXSY1,2641,26.91
BPXSY2,2404,24.50
BPXSY3,2405,24.51
BPXSY4,9298,94.75
BPXDI1,2641,26.91


In [34]:
# Removing the BPXSY4 and BPXDI4 as they have high missingess and is not always required
examination_selected.drop(columns=["BPXSY4", "BPXDI4"], inplace=True)

In [35]:
examination_selected.describe()

,SEQN,BMXBMI,BMXWAIST,BMXWT,BMXHT,BPXSY1,BPXSY2,BPXSY3,BPXDI1,BPXDI2,BPXDI3,BPXPLS
count,9813.000000,9055.000000,8661.000000,9723.000000,9067.000000,7172.000000,7409.000000,7408.000000,7172.000000,7409.000000,7408.000000,7549.000000
mean,78644.559971,25.678244,87.272047,62.599054,155.883759,118.123536,118.230530,117.999460,65.769939,65.237954,65.035637,74.416214
std,2938.592266,7.955137,22.542605,32.331616,23.176270,18.078149,18.181197,18.079852,14.960108,15.700239,16.233174,12.367969
min,73557.000000,12.100000,40.200000,3.100000,79.700000,66.000000,66.000000,62.000000,0.000000,0.000000,0.000000,40.000000
25%,76092.000000,19.700000,71.200000,37.950000,149.500000,106.000000,106.000000,106.000000,58.000000,58.000000,58.000000,66.000000
50%,78643.000000,24.700000,87.800000,65.300000,162.000000,116.000000,116.000000,114.000000,66.000000,66.000000,68.000000,74.000000
75%,81191.000000,30.200000,102.800000,83.500000,171.050000,128.000000,128.000000,128.000000,76.000000,74.000000,74.000000,82.000000
max,83731.000000,82.900000,177.900000,222.600000,202.600000,228.000000,230.000000,228.000000,122.000000,116.000000,118.000000,180.000000


## Observation

- The examination dataset originally contains **9,813 participant records and 224 variables**.
- Relevant variables were selected to examine body composition and cardiovascular health:
  1. `SEQN` – Unique participant identifier.
  2. `BMXBMI` – Body Mass Index.
  3. `BMXWAIST` – Waist circumference.
  4. `BMXWT` – Body weight.
  5. `BMXHT` – Standing height.
  6. `BPXSY1`, `BPXSY2`, `BPXSY3` – First, second, and third systolic blood pressure readings.
  7. `BPXDI1`, `BPXDI2`, `BPXDI3` – First, second, and third diastolic blood pressure readings.
  8. `BPXPLS` – Pulse rate.
- The fourth systolic and diastolic blood pressure readings (`BPXSY4` and `BPXDI4`) were excluded because approximately **94.75%** of their values were missing.
- The selected dataset contains no duplicate rows, and `SEQN` has no missing values.
- All selected health measurements are stored as numerical `float64` variables, while `SEQN` is stored as `int64`.
- Missing values vary across measurements. Weight (`BMXWT`) has the lowest missingness at approximately **0.92%**, while the first systolic and diastolic blood pressure readings each have approximately **26.91%** missing values.
- Descriptive statistics show variation in body measurements, blood pressure, and pulse rate across participants. These variables can support comparisons between participants with and without a reported cancer history.
- Missing clinical measurements are retained as `NaN` rather than automatically replaced with mean or median values, preserving the distinction between observed and unobserved measurements.

Overall, the selected examination variables provide objective measurements of body composition and cardiovascular health that can be combined with lifestyle, demographic, dietary, and laboratory variables to investigate factors associated with reported cancer history.

# Diet Dataset

The diet dataset contains dietary intake information collected from NHANES participants. 
It provides information about the amount of energy, sugars, fats, fibre, and other nutrients consumed by participants. 
For this project, relevant dietary variables will be selected to study the relationship between dietary habits and reported cancer history. 

In [36]:
diet_data.columns.to_list()

['SEQN',
 'WTDRD1',
 'WTDR2D',
 'DR1DRSTZ',
 'DR1EXMER',
 'DRABF',
 'DRDINT',
 'DR1DBIH',
 'DR1DAY',
 'DR1LANG',
 'DR1MNRSP',
 'DR1HELPD',
 'DBQ095Z',
 'DBD100',
 'DRQSPREP',
 'DR1STY',
 'DR1SKY',
 'DRQSDIET',
 'DRQSDT1',
 'DRQSDT2',
 'DRQSDT3',
 'DRQSDT4',
 'DRQSDT5',
 'DRQSDT6',
 'DRQSDT7',
 'DRQSDT8',
 'DRQSDT9',
 'DRQSDT10',
 'DRQSDT11',
 'DRQSDT12',
 'DRQSDT91',
 'DR1TNUMF',
 'DR1TKCAL',
 'DR1TPROT',
 'DR1TCARB',
 'DR1TSUGR',
 'DR1TFIBE',
 'DR1TTFAT',
 'DR1TSFAT',
 'DR1TMFAT',
 'DR1TPFAT',
 'DR1TCHOL',
 'DR1TATOC',
 'DR1TATOA',
 'DR1TRET',
 'DR1TVARA',
 'DR1TACAR',
 'DR1TBCAR',
 'DR1TCRYP',
 'DR1TLYCO',
 'DR1TLZ',
 'DR1TVB1',
 'DR1TVB2',
 'DR1TNIAC',
 'DR1TVB6',
 'DR1TFOLA',
 'DR1TFA',
 'DR1TFF',
 'DR1TFDFE',
 'DR1TCHL',
 'DR1TVB12',
 'DR1TB12A',
 'DR1TVC',
 'DR1TVD',
 'DR1TVK',
 'DR1TCALC',
 'DR1TPHOS',
 'DR1TMAGN',
 'DR1TIRON',
 'DR1TZINC',
 'DR1TCOPP',
 'DR1TSODI',
 'DR1TPOTA',
 'DR1TSELE',
 'DR1TCAFF',
 'DR1TTHEO',
 'DR1TALCO',
 'DR1TMOIS',
 'DR1TS040',
 'DR1TS060',
 'DR1TS080

## Columns Selected

- SEQN - Respondent sequence number.
- DR1TKCAL - Energy intake in kilocalories.
- DR1TPROT - Protein intake in grams.
- DR1TCARB - Carbohydrate intake in grams.
- DR1TSUGR - Total sugars intake in grams.
- DR1TFIBE - Dietary fiber intake in grams.
- DR1TTFAT - Total fat intake in grams.
- DR1TSFAT - Total saturated fat intake in grams.
- DR1TCHOL - Cholesterol intake in milligrams.
- DR1TALCO - Alcohol intake in grams.
- DR1TCAFF - Caffeine intake in milligrams.

In [37]:
diet_selected = diet_data[
    ["SEQN", "DR1TKCAL", "DR1TPROT", "DR1TCARB",
     "DR1TSUGR", "DR1TFIBE", "DR1TTFAT",
     "DR1TSFAT", "DR1TCHOL", "DR1TALCO", "DR1TCAFF"]
]

In [38]:
diet_selected.dtypes.to_frame("Data Type")

,Data Type
SEQN,int64
DR1TKCAL,float64
DR1TPROT,float64
DR1TCARB,float64
DR1TSUGR,float64
DR1TFIBE,float64
DR1TTFAT,float64
DR1TSFAT,float64
DR1TCHOL,float64
DR1TALCO,float64


In [39]:
diet_selected.isnull().sum()

SEQN           0
DR1TKCAL    1282
DR1TPROT    1282
DR1TCARB    1282
DR1TSUGR    1282
DR1TFIBE    1282
DR1TTFAT    1282
DR1TSFAT    1282
DR1TCHOL    1282
DR1TALCO    1282
DR1TCAFF    1282
dtype: int64

In [40]:
diet_selected.shape

(9813, 11)

In [41]:
diet_selected.duplicated().sum()

np.int64(0)

In [42]:
diet_selected.describe()

,SEQN,DR1TKCAL,DR1TPROT,DR1TCARB,DR1TSUGR,DR1TFIBE,DR1TTFAT,DR1TSFAT,DR1TCHOL,DR1TALCO,DR1TCAFF
count,9813.000000,8531.000000,8531.000000,8531.000000,8531.000000,8531.000000,8531.00000,8531.000000,8531.000000,8531.000000,8531.000000
mean,78644.559971,1964.538624,74.536708,242.737141,109.302640,15.278045,75.09880,24.764254,262.368655,6.092814,93.338999
std,2938.592266,984.493064,44.723181,123.956800,73.340526,10.132656,45.50421,15.883314,233.579131,22.535972,157.170984
min,73557.000000,117.000000,0.000000,8.670000,0.130000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000
25%,76092.000000,1309.500000,45.780000,158.720000,61.180000,8.500000,43.71500,13.820000,107.000000,0.000000,0.000000
50%,78643.000000,1794.000000,66.050000,223.300000,93.980000,13.100000,66.23000,21.218000,193.000000,0.000000,25.000000
75%,81191.000000,2430.000000,93.860000,301.165000,138.720000,19.800000,94.99000,31.930500,352.000000,0.000000,130.000000
max,83731.000000,12108.000000,869.490000,1423.870000,1115.500000,136.300000,548.38000,177.467000,3515.000000,591.400000,2448.000000


In [43]:
(diet_selected.iloc[:, 1:] < 0).sum() # Checking if any negative value is present, so that we can remove it or replace it

DR1TKCAL    0
DR1TPROT    0
DR1TCARB    0
DR1TSUGR    0
DR1TFIBE    0
DR1TTFAT    0
DR1TSFAT    0
DR1TCHOL    0
DR1TALCO    0
DR1TCAFF    0
dtype: int64

## Observation

- The diet dataset originally contains 9,813 participant records and 168 variables.
- For this project, 11 relevant variables were selected:
    1) `SEQN`
    2) `DR1TKCAL`
    3) `DR1TPROT`
    4) `DR1TCARB`
    5) `DR1TSUGR`
    6) `DR1TFIBE`
    7) `DR1TTFAT`
    8) `DR1TSFAT`
    9) `DR1TCHOL`
    10) `DR1TALCO`
    11) `DR1TCAFF`
- Selected variables provide information about energy, protein, carbohydrates, total sugars, dietary fibre, total fat, saturated fat, cholesterol, alcohol, and caffeine intake.
- All selected dietary measurements are stored as float64
- Each selected dietary variable contains 1,282 missing values, while SEQN contains none.
- No duplicate rows were found in the selected diet dataset.
- The descriptive statistics show substantial variation in dietary intake among participants, which provides useful numerical variables for comparing dietary patterns across cancer-history groups.
- Found no negative values in any of the selected dietary measurements

Overall, the selected dietary variables provide useful information for examining whether differences in calorie, sugar, fat, fibre, alcohol, caffeine, and other nutrient intake are associated with reported cancer history.

# Laboratory Dataset

The laboratory dataset contains measured biochemical and physiological laboratory results collected from NHANES participants. These measurements provide objective health and metabolic information that can complement the demographic, lifestyle, examination, and dietary variables in this project. Relevant laboratory variables will be selected to investigate whether differences in biomarkers are associated with reported cancer history and lifestyle-related factors.

In [44]:
labs_data.columns.to_list()

['SEQN',
 'URXUMA',
 'URXUMS',
 'URXUCR.x',
 'URXCRS',
 'URDACT',
 'WTSAF2YR.x',
 'LBXAPB',
 'LBDAPBSI',
 'LBXSAL',
 'LBDSALSI',
 'LBXSAPSI',
 'LBXSASSI',
 'LBXSATSI',
 'LBXSBU',
 'LBDSBUSI',
 'LBXSC3SI',
 'LBXSCA',
 'LBDSCASI',
 'LBXSCH',
 'LBDSCHSI',
 'LBXSCK',
 'LBXSCLSI',
 'LBXSCR',
 'LBDSCRSI',
 'LBXSGB',
 'LBDSGBSI',
 'LBXSGL',
 'LBDSGLSI',
 'LBXSGTSI',
 'LBXSIR',
 'LBDSIRSI',
 'LBXSKSI',
 'LBXSLDSI',
 'LBXSNASI',
 'LBXSOSSI',
 'LBXSPH',
 'LBDSPHSI',
 'LBXSTB',
 'LBDSTBSI',
 'LBXSTP',
 'LBDSTPSI',
 'LBXSTR',
 'LBDSTRSI',
 'LBXSUA',
 'LBDSUASI',
 'LBXWBCSI',
 'LBXLYPCT',
 'LBXMOPCT',
 'LBXNEPCT',
 'LBXEOPCT',
 'LBXBAPCT',
 'LBDLYMNO',
 'LBDMONO',
 'LBDNENO',
 'LBDEONO',
 'LBDBANO',
 'LBXRBCSI',
 'LBXHGB',
 'LBXHCT',
 'LBXMCVSI',
 'LBXMCHSI',
 'LBXMC',
 'LBXRDW',
 'LBXPLTSI',
 'LBXMPSI',
 'URXUCL',
 'WTSA2YR.x',
 'LBXSCU',
 'LBDSCUSI',
 'LBXSSE',
 'LBDSSESI',
 'LBXSZN',
 'LBDSZNSI',
 'URXUCR.y',
 'WTSB2YR.x',
 'URXBP3',
 'URDBP3LC',
 'URXBPH',
 'URDBPHLC',
 'URXBPF',
 'URDBPFLC',
 

## Columns Selected

1) SEQN - Respondent sequence number.
2) LBXSGL - Glucose, refrigerated serum (mg/dL).
3) LBXGH - Glycohemoglobin / HbA1c (%).
4) LBXSCH - Total cholesterol (mg/dL).
5) LBDHDD - Direct HDL cholesterol (mg/dL).
6) LBDLDL - LDL cholesterol (mg/dL).
7) LBXTR - Triglycerides (mg/dL).
8) LBXSATSI - Alanine aminotransferase (ALT) (U/L).
9) LBXSASSI - Aspartate aminotransferase (AST) (U/L).
10) LBXSGTSI - Gamma-glutamyl transferase (GGT) (U/L).
11) LBXSCR - Creatinine (mg/dL).
12) LBXSUA - Uric acid (mg/dL).
13) LBDSALSI - Albumin (g/L).
14) LBXLYPCT - Lymphocyte percentage (%).
15) LBXNEPCT - Segmented neutrophil percentage (%).

In [45]:
# Select the laboratory columns relevant to the project
labs_selected = labs_data[
    ["SEQN", "LBXSGL","LBXGH","LBXSCH","LBDHDD","LBDLDL",
     "LBXTR","LBXSATSI","LBXSASSI","LBXSGTSI","LBXSCR",
     "LBXSUA","LBDSALSI","LBXLYPCT","LBXNEPCT"]
]


In [46]:
labs_selected.dtypes.to_frame("Data Type")

,Data Type
SEQN,int64
LBXSGL,float64
LBXGH,float64
LBXSCH,float64
LBDHDD,float64
LBDLDL,float64
LBXTR,float64
LBXSATSI,float64
LBXSASSI,float64
LBXSGTSI,float64


In [47]:
labs_selected.isnull().sum()

SEQN           0
LBXSGL      3260
LBXGH       3170
LBXSCH      3262
LBDHDD      2189
LBDLDL      6708
LBXTR       6667
LBXSATSI    3262
LBXSASSI    3262
LBXSGTSI    3261
LBXSCR      3260
LBXSUA      3262
LBDSALSI    3260
LBXLYPCT    1294
LBXNEPCT    1294
dtype: int64

In [48]:
labs_selected.shape

(9813, 15)

In [49]:
labs_selected.duplicated().sum()

np.int64(0)

In [50]:
labs_selected.describe()

,SEQN,LBXSGL,LBXGH,LBXSCH,LBDHDD,LBDLDL,LBXTR,LBXSATSI,LBXSASSI,LBXSGTSI,LBXSCR,LBXSUA,LBDSALSI,LBXLYPCT,LBXNEPCT
count,9813.000000,6553.000000,6643.000000,6551.000000,7624.000000,3105.000000,3146.000000,6551.000000,6551.000000,6552.000000,6553.000000,6551.000000,6553.000000,8519.000000,8519.000000
mean,78644.559971,102.290859,5.642556,184.505877,53.105194,106.221256,112.306739,24.069913,25.117845,26.081197,0.880172,5.349855,42.820845,33.667261,54.491572
std,2938.592266,38.724165,1.004850,41.977990,15.230838,34.988663,115.607060,18.484877,17.740698,42.896888,0.487262,1.403767,3.436491,10.584165,11.388322
min,73557.000000,49.000000,3.500000,72.000000,10.000000,14.000000,13.000000,6.000000,9.000000,4.000000,0.290000,0.700000,24.000000,2.600000,8.400000
25%,76092.000000,86.000000,5.200000,155.000000,42.000000,81.000000,60.000000,15.000000,19.000000,13.000000,0.690000,4.300000,41.000000,26.300000,47.600000
50%,78643.000000,93.000000,5.400000,181.000000,51.000000,103.000000,88.000000,20.000000,22.000000,17.000000,0.820000,5.200000,43.000000,32.600000,55.500000
75%,81191.000000,104.000000,5.800000,210.000000,61.000000,127.000000,133.000000,27.000000,27.000000,26.000000,0.980000,6.200000,45.000000,39.700000,62.300000
max,83731.000000,577.000000,17.500000,639.000000,173.000000,375.000000,4233.000000,536.000000,882.000000,1510.000000,17.410000,13.300000,56.000000,88.000000,92.500000


In [51]:
(labs_selected.iloc[:, 1:] < 0).sum()

LBXSGL      0
LBXGH       0
LBXSCH      0
LBDHDD      0
LBDLDL      0
LBXTR       0
LBXSATSI    0
LBXSASSI    0
LBXSGTSI    0
LBXSCR      0
LBXSUA      0
LBDSALSI    0
LBXLYPCT    0
LBXNEPCT    0
dtype: int64

## Observation

## Observation

- The laboratory dataset contains **9,813 participant records and 424 variables**.
- A total of **15 relevant columns** were selected, including the participant identifier and 14 laboratory measurements.

- For this project, 15 relevant laboratory variables were selected:
    1) SEQN,
    2) LBXSGL,
    3) LBXGH,
    4) LBXSCH,
    5) LBDHDD,
    6) LBDLDL,
    7) LBXTR,
    8) LBXSATSI,
    9) LBXSASSI,
    10) LBXSGTSI,
    11) LBXSCR,
    12) LBXSUA,
    13) LBDSALSI,
    14) LBXLYPCT,
    15) LBXNEPCT.


- The selected variables cover blood glucose, HbA1c, total cholesterol, HDL, LDL, triglycerides, liver enzymes, creatinine, uric acid, albumin, and white blood cell differential percentages.
- `SEQN` is stored as `int64`, while the laboratory measurements are stored as `float64`.
- No duplicate rows were found in the selected dataset.
- Missing values vary across laboratory measurements. LDL cholesterol (`LBDLDL`) has approximately **68.36%** missing values, while triglycerides (`LBXTR`) have approximately **67.95%** missing values. These higher missingness levels should be considered during subsequent analysis.
- Lymphocyte and neutrophil percentages (`LBXLYPCT` and `LBXNEPCT`) each have approximately **13.19%** missing values.
- No negative values were detected in the selected laboratory measurements.
- Descriptive statistics indicate substantial variation in several biomarkers across participants. These measurements can be investigated for associations with reported cancer history, while accounting for missing data and potential confounding factors.
- Lymphocyte and neutrophil percentages may also be used to derive the neutrophil-to-lymphocyte ratio (NLR), provided the values are valid and non-missing.

Overall, the laboratory dataset provides useful biochemical and physiological measurements for investigating potential associations between health indicators and reported cancer history..

# Medication Dataset

The medications dataset contains information about prescription medications reported by NHANES participants. 
It can provide additional information about participants' medication use and underlying health conditions. However, unlike the other datasets, this dataset may contain multiple medication records for the same participant.

In [52]:
medications_data.columns.to_list()

['SEQN',
 'RXDUSE',
 'RXDDRUG',
 'RXDDRGID',
 'RXQSEEN',
 'RXDDAYS',
 'RXDRSC1',
 'RXDRSC2',
 'RXDRSC3',
 'RXDRSD1',
 'RXDRSD2',
 'RXDRSD3',
 'RXDCOUNT']

## Columns Selected

- SEQN - Respondent sequence number.
- RXDUSE - Whether prescription medicine was taken in the past 30 days.
- RXDDRUG - Standardized generic drug name, including listed ingredients for combination drugs.
- RXDDRGID - Generic drug code.
- RXDDAYS - Number of days the medicine was taken.
- RXDCOUNT - Number of prescription medicines reported.

In [53]:
medications_selected = medications_data[
    ["SEQN", "RXDUSE", "RXDDRUG", "RXDDRGID", "RXDDAYS", "RXDCOUNT"]
]

In [54]:
medications_selected.dtypes.to_frame("Data Type")

,Data Type
SEQN,int64
RXDUSE,int64
RXDDRUG,str
RXDDRGID,str
RXDDAYS,float64
RXDCOUNT,float64


In [55]:
medications_selected.isnull().sum()

SEQN           0
RXDUSE         0
RXDDRUG     6094
RXDDRGID    6178
RXDDAYS     6176
RXDCOUNT    6094
dtype: int64

In [56]:
medications_selected.shape

(20194, 6)

In [57]:
medications_selected.duplicated().sum()

np.int64(27)

In [58]:
medications_cleaned = medications_selected.drop_duplicates().copy()

print("Duplicate records removed:", len(medications_selected) - len(medications_cleaned))
print("Remaining records:", len(medications_cleaned))

Duplicate records removed: 27
Remaining records: 20167


In [59]:
medications_selected = medications_cleaned

In [60]:
print("Total medication records:", len(medications_selected))
print("Unique participants:", medications_selected["SEQN"].nunique())
print("Maximum medication records for one participant:",
      medications_selected["SEQN"].value_counts().max())

Total medication records: 20167
Unique participants: 10175
Maximum medication records for one participant: 23


In [61]:
medications_selected.groupby("SEQN").size().describe() # Gives information about how many medications do participants have

count    10175.000000
mean         1.982015
std          2.196279
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max         23.000000
dtype: float64

Each participant has at least one record in the medication dataset. However, this does not mean that every participant reported taking prescription medication

In [62]:
medications_selected.columns

Index(['SEQN', 'RXDUSE', 'RXDDRUG', 'RXDDRGID', 'RXDDAYS', 'RXDCOUNT'], dtype='str')

In [63]:
medications_cleaned.isnull().sum()

SEQN           0
RXDUSE         0
RXDDRUG     6094
RXDDRGID    6176
RXDDAYS     6174
RXDCOUNT    6094
dtype: int64

In [64]:
medications_cleaned["RXDDRUG"] = medications_cleaned["RXDDRUG"].str.upper()

In [65]:
medications_cleaned.head()

,SEQN,RXDUSE,RXDDRUG,RXDDRGID,RXDDAYS,RXDCOUNT
0,73557,1,99999,NaN,NaN,2.0
1,73557,1,INSULIN,d00262,1460.0,2.0
2,73558,1,GABAPENTIN,d03182,243.0,4.0
3,73558,1,INSULIN GLARGINE,d04538,365.0,4.0
4,73558,1,OLMESARTAN,d04801,14.0,4.0


In [66]:
drug_categories = {
    "Antihypertensive_Use": [
        "LISINOPRIL", "AMLODIPINE", "LOSARTAN", "METOPROLOL",
        "HYDROCHLOROTHIAZIDE", "FUROSEMIDE", "ATENOLOL", "CARVEDILOL",
        "CLONIDINE"
    ],
    
    "Lipid_Lowering_Use": [
        "SIMVASTATIN", "ATORVASTATIN", "PRAVASTATIN", "ROSUVASTATIN"
    ],
    
    "Antidiabetic_Use": [
        "METFORMIN", "INSULIN GLARGINE"
    ],
    
    "Thyroid_Medication_Use": [
        "LEVOTHYROXINE"
    ],
    
    "Respiratory_Medication_Use": [
        "ALBUTEROL", "MONTELUKAST"
    ],
    
    "Antidepressant_Use": [
        "SERTRALINE", "CITALOPRAM"
    ],
    
    "Anticoagulant_Antiplatelet_Use": [
        "WARFARIN", "CLOPIDOGREL"
    ],
    
    "GI_Medication_Use": [
        "OMEPRAZOLE", "RANITIDINE"
    ],
    
    "Analgesic_Use": [
        "IBUPROFEN", "ACETAMINOPHEN", "HYDROCODONE"
    ],
    
    "Antibiotic_Use": [
        "AMOXICILLIN"
    ]
}

In [67]:
def check_category(drug_name, drug_list):
    if pd.isna(drug_name):
        return 2

    drug_name = str(drug_name).strip().upper()

    if drug_name in ["55555", "77777", "99999"]:
        return 2

    for medicine in drug_list:
        if medicine in drug_name:
            return 1

    return 0

In [68]:
medications_new = medications_cleaned.copy()
for drug_cat, drugs in drug_categories.items():
    medications_new[drug_cat] = medications_cleaned["RXDDRUG"].apply(
        lambda x: check_category(x, drugs)
    )

In [69]:
medications_new[["RXDDRUG"] + list(drug_categories.keys())].head(20)

,RXDDRUG,Antihypertensive_Use,Lipid_Lowering_Use,Antidiabetic_Use,Thyroid_Medication_Use,Respiratory_Medication_Use,Antidepressant_Use,Anticoagulant_Antiplatelet_Use,GI_Medication_Use,Analgesic_Use,Antibiotic_Use
0,99999,2,2,2,2,2,2,2,2,2,2
1,INSULIN,0,0,0,0,0,0,0,0,0,0
2,GABAPENTIN,0,0,0,0,0,0,0,0,0,0
3,INSULIN GLARGINE,0,0,1,0,0,0,0,0,0,0
4,OLMESARTAN,0,0,0,0,0,0,0,0,0,0
5,SIMVASTATIN,0,1,0,0,0,0,0,0,0,0
6,INSULIN ASPART,0,0,0,0,0,0,0,0,0,0
7,INSULIN GLARGINE,0,0,1,0,0,0,0,0,0,0
8,PANCRELIPASE,0,0,0,0,0,0,0,0,0,0
9,SIMVASTATIN,0,1,0,0,0,0,0,0,0,0


In [70]:
# Count how many medication records fall into each category

for column in drug_categories.keys():
    print(column, ":", medications_new[column].sum())

Antihypertensive_Use : 14855
Lipid_Lowering_Use : 13415
Antidiabetic_Use : 12950
Thyroid_Medication_Use : 12733
Respiratory_Medication_Use : 12862
Antidepressant_Use : 12662
Anticoagulant_Antiplatelet_Use : 12571
GI_Medication_Use : 12903
Analgesic_Use : 12782
Antibiotic_Use : 12501


In [71]:
def aggregate_category(values):
    if (values == 1).any():
        return 1

    if (values == 2).any():
        return 2

    return 0

In [72]:
category_columns = list(drug_categories.keys())

aggregation_rules = {
    "RXDCOUNT": "first",
    "RXDUSE": "first",
    **{
        column: aggregate_category
        for column in category_columns
    }
}

medications_participant = (
    medications_new
    .groupby("SEQN")
    .agg(aggregation_rules)
    .reset_index()
)

# Explicitly handle participants who reported no medication
no_medication = medications_participant["RXDUSE"] == 2

medications_participant.loc[
    no_medication, category_columns
] = 0

# Handle refused and don't-know responses
unknown_use = medications_participant["RXDUSE"].isin([7, 9])

medications_participant.loc[
    unknown_use, category_columns
] = 2

In [73]:
for column in medications_participant.columns[2:]:
    print(column, ":", medications_participant[column].sum())

RXDUSE : 16296
Antihypertensive_Use : 1796
Lipid_Lowering_Use : 1213
Antidiabetic_Use : 707
Thyroid_Medication_Use : 545
Respiratory_Medication_Use : 606
Antidepressant_Use : 480
Anticoagulant_Antiplatelet_Use : 381
GI_Medication_Use : 706
Analgesic_Use : 560
Antibiotic_Use : 314


In [74]:
medications_participant.head()

,SEQN,RXDCOUNT,RXDUSE,Antihypertensive_Use,Lipid_Lowering_Use,Antidiabetic_Use,Thyroid_Medication_Use,Respiratory_Medication_Use,Antidepressant_Use,Anticoagulant_Antiplatelet_Use,GI_Medication_Use,Analgesic_Use,Antibiotic_Use
0,73557,2.0,1,2,2,2,2,2,2,2,2,2,2
1,73558,4.0,1,0,1,1,0,0,0,0,0,0,0
2,73559,5.0,1,0,1,1,0,0,0,0,0,0,0
3,73560,NaN,2,0,0,0,0,0,0,0,0,0,0
4,73561,4.0,1,1,2,2,1,2,2,2,2,2,2


In [75]:
medications_participant.shape

(10175, 13)

In [76]:
# Confirming that a record with a unique SEQN appears only once
print("Unique participants:", medications_participant["SEQN"].nunique())
print("Duplicates:",medications_participant["SEQN"].duplicated().sum())

Unique participants: 10175
Duplicates: 0


In [77]:
for column in drug_categories.keys():
    print(f"\n{column}:")
    print(
        medications_new[column]
        .value_counts()
        .reindex([0, 1, 2], fill_value=0)
    )


Antihypertensive_Use:
Antihypertensive_Use
0    11488
1     2503
2     6176
Name: count, dtype: int64

Lipid_Lowering_Use:
Lipid_Lowering_Use
0    12928
1     1063
2     6176
Name: count, dtype: int64

Antidiabetic_Use:
Antidiabetic_Use
0    13393
1      598
2     6176
Name: count, dtype: int64

Thyroid_Medication_Use:
Thyroid_Medication_Use
0    13610
1      381
2     6176
Name: count, dtype: int64

Respiratory_Medication_Use:
Respiratory_Medication_Use
0    13481
1      510
2     6176
Name: count, dtype: int64

Antidepressant_Use:
Antidepressant_Use
0    13681
1      310
2     6176
Name: count, dtype: int64

Anticoagulant_Antiplatelet_Use:
Anticoagulant_Antiplatelet_Use
0    13772
1      219
2     6176
Name: count, dtype: int64

GI_Medication_Use:
GI_Medication_Use
0    13440
1      551
2     6176
Name: count, dtype: int64

Analgesic_Use:
Analgesic_Use
0    13561
1      430
2     6176
Name: count, dtype: int64

Antibiotic_Use:
Antibiotic_Use
0    13842
1      149
2     6176
Name: co

RXDCOUNT have 6094 missing values, which correspond to participants for whom a medication count was not reported

In [78]:
# Check RXDUSE values for records where RXDCOUNT is missing
print("RXDUSE distribution:")
print(medications_participant["RXDUSE"].value_counts().sort_index())

for column in category_columns:
    print(f"\n{column}:")
    print(
        medications_participant[column]
        .value_counts()
        .reindex([0, 1, 2], fill_value=0)
    )

RXDUSE distribution:
RXDUSE
1    4081
2    6089
7       4
9       1
Name: count, dtype: int64

Antihypertensive_Use:
Antihypertensive_Use
0    8447
1    1660
2      68
Name: count, dtype: int64

Lipid_Lowering_Use:
Lipid_Lowering_Use
0    9039
1    1059
2      77
Name: count, dtype: int64

Antidiabetic_Use:
Antidiabetic_Use
0    9546
1     551
2      78
Name: count, dtype: int64

Thyroid_Medication_Use:
Thyroid_Medication_Use
0    9712
1     381
2      82
Name: count, dtype: int64

Respiratory_Medication_Use:
Respiratory_Medication_Use
0    9652
1     440
2      83
Name: count, dtype: int64

Antidepressant_Use:
Antidepressant_Use
0    9781
1     308
2      86
Name: count, dtype: int64

Anticoagulant_Antiplatelet_Use:
Anticoagulant_Antiplatelet_Use
0    9878
1     213
2      84
Name: count, dtype: int64

GI_Medication_Use:
GI_Medication_Use
0    9551
1     542
2      82
Name: count, dtype: int64

Analgesic_Use:
Analgesic_Use
0    9699
1     392
2      84
Name: count, dtype: int64

Antib

- RXDUSE = 2 - participant reported no prescription medicine, so RXDCOUNT should be 0.
- RXDUSE = 7 - refused → keep as missing.
- RXDUSE = 9 - don't know → keep as missing.

In [79]:
# Set medication count to 0 only when the participant reported no medication
medications_participant.loc[medications_participant["RXDUSE"] == 2,"RXDCOUNT"] = 0

In [80]:
medications_participant["RXDCOUNT"].isnull().sum()

np.int64(5)

Now only 5 RXDCOUNT values remain missing, corresponding to the RXDUSE = 7 (refused) and RXDUSE = 9 (don't know) responses.

In [81]:
medications_participant.head()

,SEQN,RXDCOUNT,RXDUSE,Antihypertensive_Use,Lipid_Lowering_Use,Antidiabetic_Use,Thyroid_Medication_Use,Respiratory_Medication_Use,Antidepressant_Use,Anticoagulant_Antiplatelet_Use,GI_Medication_Use,Analgesic_Use,Antibiotic_Use
0,73557,2.0,1,2,2,2,2,2,2,2,2,2,2
1,73558,4.0,1,0,1,1,0,0,0,0,0,0,0
2,73559,5.0,1,0,1,1,0,0,0,0,0,0,0
3,73560,0.0,2,0,0,0,0,0,0,0,0,0,0
4,73561,4.0,1,1,2,2,1,2,2,2,2,2,2


In [82]:
medications_participant.describe()

,SEQN,RXDCOUNT,RXDUSE,Antihypertensive_Use,Lipid_Lowering_Use,Antidiabetic_Use,Thyroid_Medication_Use,Respiratory_Medication_Use,Antidepressant_Use,Anticoagulant_Antiplatelet_Use,GI_Medication_Use,Analgesic_Use,Antibiotic_Use
count,10175.000000,10170.000000,10175.000000,10175.000000,10175.000000,10175.000000,10175.000000,10175.000000,10175.000000,10175.000000,10175.000000,10175.000000,10175.000000
mean,78644.000000,1.386431,1.601572,0.176511,0.119214,0.069484,0.053563,0.059558,0.047174,0.037445,0.069386,0.055037,0.030860
std,2937.413829,2.503873,0.506957,0.398418,0.346625,0.282835,0.258492,0.268947,0.248715,0.229257,0.284073,0.261774,0.215004
min,73557.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,76100.500000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,78644.000000,0.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,81187.500000,2.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,83731.000000,23.000000,9.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000


## Observation

- Original medications dataset contained 20,194 medication records for 10,175 participants.
- After removing exact duplicate records from the selected medication data, 20,167 records remained.
- Multiple medication records can belong to the same participant, with some participants having several medication records.
- Medication data was therefore converted from a record-level dataset to a participant-level dataset using SEQN.
- Final participant-level medication dataset contains 10,175 participants, with one row per participant.
- RXDUSE indicates whether prescription medication was used, while RXDCOUNT represents the participant's reported number of prescription medicines.
- For participants with RXDUSE = 2, RXDCOUNT was set to 0, because they reported no prescription medication.
- The remaining 5 missing RXDCOUNT values correspond to refused or don't-know responses and were retained as missing.
- Ten new binary medication-category columns were created:
    1) Antihypertensive_Use
    2) Lipid_Lowering_Use
    3) Antidiabetic_Use
    4) Thyroid_Medication_Use
    5) Respiratory_Medication_Use
    6) Antidepressant_Use
    7) Anticoagulant_Antiplatelet_Use
    8) GI_Medication_Use
    9) Analgesic_Use
    10) Antibiotic_Use
- Each category uses 1 when at least one matching medication was reported and 0 otherwise.
- Category variables contain no missing values, and each participant appears only once in the final dataset.

Overall, the medication dataset has been converted into a useful participant-level feature set that can be merged with the demographic, questionnaire, examination, diet, and laboratory datasets using SEQN.
Important limitation: the medication categories are based on the specific drug names included in our manually defined lists, so they represent matched medication categories, not a complete clinical classification of every possible medication in NHANES.

# Creating The Final Dataset

## Merging Final DataFrames

In [83]:
# Check that SEQN is unique in each participant-level dataset
final_datasets = {
    "Demographic": demographic_selected,
    "Questionnaire": questionnaire_cleaned,
    "Examination": examination_selected,
    "Diet": diet_selected,
    "Laboratory": labs_selected,
    "Medications": medications_participant
}

dataset_summary = pd.DataFrame(
    {
        "Rows": [i.shape[0] for i in final_datasets.values()],
        "Columns": [i.shape[1] for i in final_datasets.values()],
        "Unique Values": [i["SEQN"].nunique() for i in final_datasets.values()],
        "Duplicate Values": [i["SEQN"].duplicated().sum() for i in final_datasets.values()]
    }
)
dataset_summary["Dataset"] = [i for i in final_datasets.keys()]
dataset_summary.set_index("Dataset", inplace=True)

In [84]:
dataset_summary

,Rows,Columns,Unique Values,Duplicate Values
Dataset,,,,
Demographic,10175,5,10175,0
Questionnaire,10175,11,10175,0
Examination,9813,12,9813,0
Diet,9813,11,9813,0
Laboratory,9813,15,9813,0
Medications,10175,13,10175,0


In [85]:
# Merge all participant-level datasets using SEQN
final_dataset = demographic_selected.merge(
    questionnaire_cleaned,
    on="SEQN",
    how="left"
).merge(
    examination_selected,
    on="SEQN",
    how="left"
).merge(
    diet_selected,
    on="SEQN",
    how="left"
).merge(
    labs_selected,
    on="SEQN",
    how="left"
).merge(
    medications_participant,
    on="SEQN",
    how="left"
)

In [86]:
print("Final dataset shape:", final_dataset.shape)
display(final_dataset.head())

Final dataset shape: (10175, 62)


,SEQN,RIAGENDR,RIDAGEYR,RIDRETH1,INDFMPIR,MCQ220,SMQ020,SMQ040,SMD650,ALQ101,...,Antihypertensive_Use,Lipid_Lowering_Use,Antidiabetic_Use,Thyroid_Medication_Use,Respiratory_Medication_Use,Antidepressant_Use,Anticoagulant_Antiplatelet_Use,GI_Medication_Use,Analgesic_Use,Antibiotic_Use
0,73557,1,69,4,0.84,2.0,1.0,3.0,NaN,1.0,...,2,2,2,2,2,2,2,2,2,2
1,73558,1,54,3,1.78,2.0,1.0,2.0,1.0,1.0,...,0,1,1,0,0,0,0,0,0,0
2,73559,1,72,3,4.51,1.0,1.0,3.0,NaN,1.0,...,0,1,1,0,0,0,0,0,0,0
3,73560,1,9,3,2.52,NaN,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0
4,73561,2,73,3,5.00,2.0,2.0,NaN,NaN,1.0,...,1,2,2,1,2,2,2,2,2,2


In [87]:
# Verify the final merged dataset
print("Rows:", len(final_dataset))
print("Columns:", len(final_dataset.columns))
print("Unique participants:", final_dataset["SEQN"].nunique())
print("Duplicate rows:", final_dataset.duplicated().sum())
print("Duplicate SEQN:", final_dataset["SEQN"].duplicated().sum())

Rows: 10175
Columns: 62
Unique participants: 10175
Duplicate rows: 0
Duplicate SEQN: 0


In [88]:
final_dataset.columns

Index(['SEQN', 'RIAGENDR', 'RIDAGEYR', 'RIDRETH1', 'INDFMPIR', 'MCQ220',
       'SMQ020', 'SMQ040', 'SMD650', 'ALQ101', 'ALQ120Q', 'ALQ120U', 'PAQ650',
       'PAQ620', 'PAQ635', 'BMXBMI', 'BMXWAIST', 'BMXWT', 'BMXHT', 'BPXSY1',
       'BPXSY2', 'BPXSY3', 'BPXDI1', 'BPXDI2', 'BPXDI3', 'BPXPLS', 'DR1TKCAL',
       'DR1TPROT', 'DR1TCARB', 'DR1TSUGR', 'DR1TFIBE', 'DR1TTFAT', 'DR1TSFAT',
       'DR1TCHOL', 'DR1TALCO', 'DR1TCAFF', 'LBXSGL', 'LBXGH', 'LBXSCH',
       'LBDHDD', 'LBDLDL', 'LBXTR', 'LBXSATSI', 'LBXSASSI', 'LBXSGTSI',
       'LBXSCR', 'LBXSUA', 'LBDSALSI', 'LBXLYPCT', 'LBXNEPCT', 'RXDCOUNT',
       'RXDUSE', 'Antihypertensive_Use', 'Lipid_Lowering_Use',
       'Antidiabetic_Use', 'Thyroid_Medication_Use',
       'Respiratory_Medication_Use', 'Antidepressant_Use',
       'Anticoagulant_Antiplatelet_Use', 'GI_Medication_Use', 'Analgesic_Use',
       'Antibiotic_Use'],
      dtype='str')

## Final Dataset Columns Description

### Demographic

- `SEQN` — Unique respondent sequence number.
- `RIAGENDR` — Participant's sex.
- `RIDAGEYR` — Age in years.
- `RIDRETH1` — Race and ethnicity category.
- `INDFMPIR` — Family income-to-poverty ratio.

### Questionnaire

- `MCQ220` — Whether the participant has ever been told they had cancer.
- `SMQ020` — Whether the participant has smoked at least 100 cigarettes in their lifetime.
- `SMQ040` — Current smoking status among eligible participants.
- `SMD650` — Number of cigarettes smoked per day.
- `ALQ101` — Alcohol consumption history.
- `ALQ120Q` — Quantity or frequency component of alcohol consumption, as represented in the source dataset.
- `ALQ120U` — Unit associated with the alcohol consumption frequency or quantity.
- `PAQ650` — Vigorous recreational physical activity.
- `PAQ620` — Moderate work-related physical activity.
- `PAQ635` — Walking or bicycling for transportation.

### Examination

- `BMXBMI` — Body Mass Index (BMI).
- `BMXWAIST` — Waist circumference.
- `BMXWT` — Body weight.
- `BMXHT` — Standing height.
- `BPXSY1` — First systolic blood pressure reading.
- `BPXSY2` — Second systolic blood pressure reading.
- `BPXSY3` — Third systolic blood pressure reading.
- `BPXDI1` — First diastolic blood pressure reading.
- `BPXDI2` — Second diastolic blood pressure reading.
- `BPXDI3` — Third diastolic blood pressure reading.
- `BPXPLS` — Pulse rate.

### Diet

- `DR1TKCAL` — Total energy intake.
- `DR1TPROT` — Total protein intake.
- `DR1TCARB` — Total carbohydrate intake.
- `DR1TSUGR` — Total sugar intake.
- `DR1TFIBE` — Total dietary fibre intake.
- `DR1TTFAT` — Total fat intake.
- `DR1TSFAT` — Total saturated fat intake.
- `DR1TCHOL` — Total dietary cholesterol intake.
- `DR1TALCO` — Total alcohol intake from the first-day dietary recall.
- `DR1TCAFF` — Total caffeine intake.

### Laboratory

- `LBXSGL` — Blood glucose.
- `LBXGH` — Glycohemoglobin (HbA1c).
- `LBXSCH` — Total cholesterol.
- `LBDHDD` — HDL cholesterol.
- `LBDLDL` — LDL cholesterol.
- `LBXTR` — Triglycerides.
- `LBXSATSI` — Alanine aminotransferase (ALT).
- `LBXSASSI` — Aspartate aminotransferase (AST).
- `LBXSGTSI` — Gamma-glutamyl transferase (GGT).
- `LBXSCR` — Serum creatinine.
- `LBXSUA` — Uric acid.
- `LBDSALSI` — Serum albumin.
- `LBXLYPCT` — Lymphocyte percentage.
- `LBXNEPCT` — Neutrophil percentage.

### Medications

- `RXDCOUNT` — Number of prescription medication records associated with a participant.
- `RXDUSE` — Prescription medication use indicator.
- `Antihypertensive_Use` — Indicator for selected antihypertensive medications.
- `Lipid_Lowering_Use` — Indicator for selected lipid-lowering medications.
- `Antidiabetic_Use` — Indicator for selected antidiabetic medications.
- `Thyroid_Medication_Use` — Indicator for selected thyroid medications.
- `Respiratory_Medication_Use` — Indicator for selected respiratory medications.
- `Antidepressant_Use` — Indicator for selected antidepressants.
- `Anticoagulant_Antiplatelet_Use` — Indicator for selected anticoagulant or antiplatelet medications.
- `GI_Medication_Use` — Indicator for selected gastrointestinal medications.
- `Analgesic_Use` — Indicator for selected analgesics.
- `Antibiotic_Use` — Indicator for selected antibiotics.

### Temporary Preprocessing Variable

- `AgeGroup` — Age category created for group-wise missing-value analysis or imputation. It is removed later and is not part of the final merged dataset.


In [89]:
final_dataset.to_csv("final_dataset.csv", index=False)

In [90]:
graph_names = {
    "SEQN": "Participant ID",
    "RIAGENDR": "Sex",
    "RIDAGEYR": "Age",
    "RIDRETH1": "Race/Ethnicity",
    "INDFMPIR": "Income-to-Poverty Ratio",

    "MCQ220": "Cancer History",

    "SMQ020": "Ever Smoked",
    "SMQ040": "Current Smoking",
    "SMD650": "Cigarettes per Day",

    "ALQ101": "Alcohol Use History",
    "ALQ120Q": "Alcohol Frequency",
    "ALQ120U": "Alcohol Frequency Unit",

    "PAQ650": "Vigorous Activity",
    "PAQ620": "Moderate Work Activity",
    "PAQ635": "Walking/Cycling Activity",

    "BMXBMI": "BMI",
    "BMXWAIST": "Waist Circumference",
    "BPXSY1": "Systolic BP",
    "BPXDI1": "Diastolic BP",
    "BPXPLS": "Pulse Rate",
    "BMXWT": "Body Weight",
    "BMXHT": "Height",
    "BPXSY2": "Systolic BP Reading 2",
    "BPXSY3": "Systolic BP Reading 3",
    "BPXDI2": "Diastolic BP Reading 2",
    "BPXDI3": "Diastolic BP Reading 3",

    "DR1TKCAL": "Energy Intake",
    "DR1TPROT": "Protein Intake",
    "DR1TCARB": "Carbohydrate Intake",
    "DR1TSUGR": "Sugar Intake",
    "DR1TFIBE": "Fiber Intake",
    "DR1TTFAT": "Total Fat",
    "DR1TSFAT": "Saturated Fat",
    "DR1TCHOL": "Cholesterol Intake",
    "DR1TALCO": "Dietary Alcohol",
    "DR1TCAFF": "Caffeine Intake",

    "LBXSGL": "Blood Glucose",
    "LBXGH": "HbA1c",
    "LBXSCH": "Total Cholesterol",
    "LBDHDD": "HDL Cholesterol",
    "LBDLDL": "LDL Cholesterol",
    "LBXTR": "Triglycerides",
    "LBXSATSI": "ALT",
    "LBXSASSI": "AST",
    "LBXSGTSI": "GGT",
    "LBXSCR": "Creatinine",
    "LBXSUA": "Uric Acid",
    "LBDSALSI": "Albumin",
    "LBXLYPCT": "Lymphocyte %",
    "LBXNEPCT": "Neutrophil %",

    "RXDCOUNT": "Prescription Count",
    "RXDUSE": "Prescription Use",

    "Antihypertensive_Use": "Antihypertensive Use",
    "Lipid_Lowering_Use": "Lipid-Lowering Use",
    "Antidiabetic_Use": "Antidiabetic Use",
    "Thyroid_Medication_Use": "Thyroid Medication Use",
    "Respiratory_Medication_Use": "Respiratory Medication Use",
    "Antidepressant_Use": "Antidepressant Use",
    "Anticoagulant_Antiplatelet_Use": "Anticoagulant/Antiplatelet Use",
    "GI_Medication_Use": "GI Medication Use",
    "Analgesic_Use": "Analgesic Use",
    "Antibiotic_Use": "Antibiotic Use"
}

In [91]:
column_descriptions = {
    # Demographic
    "SEQN": "Respondent sequence number.",
    "RIAGENDR": "Gender.",
    "RIDAGEYR": "Age in years.",
    "RIDRETH1": "Race / ethnicity.",
    "INDFMPIR": "Family income-to-poverty ratio.",

    # Questionnaire
    "MCQ220": "Reported history of cancer.",
    "SMQ020": "Ever smoked 100 or more cigarettes.",
    "SMQ040": "Current smoking status.",
    "SMD650": "Cigarettes smoked per day.",
    "ALQ101": "History of alcohol consumption.",
    "ALQ120Q": "Alcohol consumption frequency.",
    "ALQ120U": "Unit associated with the alcohol consumption frequency or quantity.",
    "PAQ650": "Vigorous recreational activity.",
    "PAQ620": "Moderate work-related activity.",
    "PAQ635": "Walking / bicycling for transportation.",

    # Examination
    "BMXBMI": "Body Mass Index.",
    "BMXWAIST": "Waist circumference.",
    "BPXSY1": "Systolic blood pressure.",
    "BPXDI1": "Diastolic blood pressure.",
    "BPXPLS": "Pulse rate.",
    "BMXWT": "Body weight.",
    "BMXHT": "Standing height.",
    "BPXSY2": "Second systolic blood pressure reading.",
    "BPXSY3": "Third systolic blood pressure reading.",
    "BPXDI2": "Second diastolic blood pressure reading.",
    "BPXDI3": "Third diastolic blood pressure reading.",

    # Diet
    "DR1TKCAL": "Energy intake.",
    "DR1TPROT": "Protein intake.",
    "DR1TCARB": "Carbohydrate intake.",
    "DR1TSUGR": "Total sugar intake.",
    "DR1TFIBE": "Dietary fibre intake.",
    "DR1TTFAT": "Total fat intake.",
    "DR1TSFAT": "Saturated fat intake.",
    "DR1TCHOL": "Cholesterol intake.",
    "DR1TALCO": "Alcohol intake.",
    "DR1TCAFF": "Caffeine intake.",

    # Laboratory
    "LBXSGL": "Blood glucose.",
    "LBXGH": "HbA1c.",
    "LBXSCH": "Total cholesterol.",
    "LBDHDD": "HDL cholesterol.",
    "LBDLDL": "LDL cholesterol.",
    "LBXTR": "Triglycerides.",
    "LBXSATSI": "ALT.",
    "LBXSASSI": "AST.",
    "LBXSGTSI": "GGT.",
    "LBXSCR": "Creatinine.",
    "LBXSUA": "Uric acid.",
    "LBDSALSI": "Albumin.",
    "LBXLYPCT": "Lymphocyte percentage.",
    "LBXNEPCT": "Neutrophil percentage.",

    # Medications
    "RXDCOUNT": "Number of prescription medicines.",
    "RXDUSE": "Prescription medicine use.",
    "Antihypertensive_Use": "Antihypertensive medication use.",
    "Lipid_Lowering_Use": "Lipid-lowering medication use.",
    "Antidiabetic_Use": "Antidiabetic medication use.",
    "Thyroid_Medication_Use": "Thyroid medication use.",
    "Respiratory_Medication_Use": "Respiratory medication use.",
    "Antidepressant_Use": "Antidepressant medication use.",
    "Anticoagulant_Antiplatelet_Use": "Anticoagulant / antiplatelet use.",
    "GI_Medication_Use": "Gastrointestinal medication use.",
    "Analgesic_Use": "Analgesic medication use.",
    "Antibiotic_Use": "Antibiotic medication use.",

    "AgeGroup": "Age group to which participant belongs"
}

# Data Preprocessing

In [92]:
processed_data = final_dataset.copy()

## Keeping only Eligible Participants (age 20+)

In [93]:
before_rows = len(processed_data)
ineligible_index = processed_data[processed_data["RIDAGEYR"] < 20].index
processed_data.drop(index=ineligible_index, inplace=True)

after_rows = len(processed_data)

print("Rows before filtering:", before_rows)
print("Ineligible rows removed:", before_rows - after_rows)
print("Rows after filtering:", after_rows)

Rows before filtering: 10175
Ineligible rows removed: 4406
Rows after filtering: 5769


In [94]:
print("Ineligible Records:",len(ineligible_index))
print("Eligible Records:", len(processed_data))

Ineligible Records: 4406
Eligible Records: 5769


## Creating AgeGroup Column for Group-Wise Imputation (if required)

In [95]:
processed_data["AgeGroup"] = pd.cut(processed_data["RIDAGEYR"], bins=[19, 29, 39, 49, 59, 69, 79, 120],labels=["20-29", "30-39", "40-49", "50-59","60-69", "70-79", "80+"])

## Analyzing Missing Values

In [96]:
missing_data = processed_data.isnull().sum()

In [97]:
# Generating summary of missing values
missing_stats = pd.DataFrame()

missing_stats["Missing_Values"] = missing_data
missing_stats["Missing_Values_Proportion(%)"] = (missing_data[missing_data > 0]/processed_data.shape[0]) * 100
req_columns = missing_stats.index.to_list()
description_col = []
for i in req_columns:
    description_col.append(column_descriptions[i])
missing_stats["Description"] = description_col
missing_stats.sort_values(ascending=False, inplace=True, by="Missing_Values_Proportion(%)")

In [98]:
# Keeping only the column names who have null values
missing_stats = missing_stats[missing_stats["Missing_Values"]>0]

In [99]:
missing_stats.head()

,Missing_Values,Missing_Values_Proportion(%),Description
SMD650,4590,79.563183,Cigarettes smoked per day.
LBDLDL,3256,56.439591,LDL cholesterol.
SMQ040,3239,56.144912,Current smoking status.
LBXTR,3216,55.746230,Triglycerides.
ALQ120U,2321,40.232276,Unit associated with the alcohol consumption f...


In [100]:
missing_stats.tail()

,Missing_Values,Missing_Values_Proportion(%),Description
BMXHT,239,4.142832,Standing height.
BMXWT,236,4.090830,Body weight.
RXDCOUNT,4,0.069336,Number of prescription medicines.
PAQ620,3,0.052002,Moderate work-related activity.
SMQ020,2,0.034668,Ever smoked 100 or more cigarettes.


In [101]:
missing_data.head()

SEQN          0
RIAGENDR      0
RIDAGEYR      0
RIDRETH1      0
INDFMPIR    455
dtype: int64

In [102]:
missing_data[missing_data > 0].sort_values(ascending=False)

SMD650      4590
LBDLDL      3256
SMQ040      3239
LBXTR       3216
ALQ120U     2321
ALQ120Q     1452
DR1TTFAT     722
DR1TSFAT     722
DR1TCHOL     722
DR1TKCAL     722
DR1TSUGR     722
DR1TFIBE     722
DR1TALCO     722
DR1TCAFF     722
DR1TCARB     722
DR1TPROT     722
ALQ101       675
BPXDI1       658
BPXSY1       658
BMXWAIST     499
BPXDI2       471
BPXSY2       471
BPXSY3       462
BPXDI3       462
INDFMPIR     455
LBXSUA       443
LBXSATSI     443
LBXSCH       443
LBXSASSI     443
LBXSGTSI     442
LBXSGL       441
LBDSALSI     441
LBXSCR       441
LBDHDD       427
LBXLYPCT     390
LBXNEPCT     390
LBXGH        377
BPXPLS       365
BMXBMI       249
BMXHT        239
BMXWT        236
RXDCOUNT       4
PAQ620         3
SMQ020         2
dtype: int64

In [103]:
missing_data.head()

SEQN          0
RIAGENDR      0
RIDAGEYR      0
RIDRETH1      0
INDFMPIR    455
dtype: int64

In [104]:
def missing_category(percent):
    if percent >= 50:
        return "High missingness"
    elif percent >= 20:
        return "Moderate missingness"
    elif percent > 0:
        return "Low missingness"
    else:
        return "No missing values"


missing_stats["Category"] = missing_stats["Missing_Values_Proportion(%)"].apply(missing_category)

In [105]:
missing_stats

,Missing_Values,Missing_Values_Proportion(%),Description,Category
SMD650,4590,79.563183,Cigarettes smoked per day.,High missingness
LBDLDL,3256,56.439591,LDL cholesterol.,High missingness
SMQ040,3239,56.144912,Current smoking status.,High missingness
LBXTR,3216,55.746230,Triglycerides.,High missingness
ALQ120U,2321,40.232276,Unit associated with the alcohol consumption f...,Moderate missingness
ALQ120Q,1452,25.169007,Alcohol consumption frequency.,Moderate missingness
DR1TTFAT,722,12.515167,Total fat intake.,Low missingness
DR1TSFAT,722,12.515167,Saturated fat intake.,Low missingness
DR1TCHOL,722,12.515167,Cholesterol intake.,Low missingness
DR1TKCAL,722,12.515167,Energy intake.,Low missingness


## Analysis of High Missing Value Columns

In [106]:
display(missing_stats.head(4))

,Missing_Values,Missing_Values_Proportion(%),Description,Category
SMD650,4590,79.563183,Cigarettes smoked per day.,High missingness
LBDLDL,3256,56.439591,LDL cholesterol.,High missingness
SMQ040,3239,56.144912,Current smoking status.,High missingness
LBXTR,3216,55.746230,Triglycerides.,High missingness


In [107]:
print(column_descriptions["SMD650"])
print(column_descriptions["SMQ040"])
print(column_descriptions["LBDLDL"])
print(column_descriptions["LBXTR"])

Cigarettes smoked per day.
Current smoking status.
LDL cholesterol.
Triglycerides.


In [108]:
# Descriptive statistics for continuous variables
continuous_high_missing = ["SMD650", "LBDLDL", "LBXTR"]

display(processed_data[continuous_high_missing].describe())

# Frequency distribution for categorical variable
print("SMQ040 - Current Smoking Status")
display(processed_data["SMQ040"].value_counts(dropna=False).sort_index())

,SMD650,LBDLDL,LBXTR
count,1179.000000,2513.000000,2553.000000
mean,10.648855,110.958217,120.858598
std,8.500506,35.169871,124.672596
min,1.000000,14.000000,14.000000
25%,4.000000,86.000000,65.000000
50%,10.000000,108.000000,95.000000
75%,15.000000,133.000000,144.000000
max,90.000000,375.000000,4233.000000


SMQ040 - Current Smoking Status


SMQ040
1.0     965
2.0     229
3.0    1336
NaN    3239
Name: count, dtype: int64

In [109]:
processed_data.dtypes["SMQ040"]

dtype('float64')

### SMD650 — Cigarettes Smoked per Day

- `SMD650` records the number of cigarettes smoked per day among applicable participants.
- Approximately **79.56%** of the values are missing (4,590 out of 5,769 participants).
- Among the observed values, the mean is **10.65 cigarettes per day**, with a standard deviation of **8.50**.
- The median is **10 cigarettes per day**, while the maximum is **90 cigarettes per day**.
- Due to its very high missingness, `SMD650` is excluded from the dataset to avoid relying on a variable with limited available observations.

### SMQ040 — Current Smoking Status

- `SMQ040` is a categorical variable describing current smoking status among eligible participants.
- Its responses represent:
  - `1` — Every day
  - `2` — Some days
  - `3` — Not at all
- Approximately **56.14%** of its values are missing (3,239 out of 5,769 participants).
- The mean and standard deviation are not meaningful for this categorical variable, even though it is stored numerically.
- Some missing values may result from questionnaire skip patterns or participant non-response.
- The variable is retained because it provides useful information about current smoking behaviour.
- Missing values will not be replaced using numerical mean or median imputation.

### LBDLDL — LDL Cholesterol

- The mean LDL cholesterol level is **110.96 mg/dL**, with a standard deviation of **35.17 mg/dL**.
- The median is **108 mg/dL**, while the maximum observed value is **375 mg/dL**.
- Approximately **56.44%** of the values are missing (3,256 out of 5,769 participants).
- The observed values show considerable variation across participants.
- The maximum value should not be removed automatically, as it may represent a genuine laboratory measurement.
- Due to the high missingness, mean or median imputation is not appropriate at this stage.
- The variable is retained, and its missing values will be handled appropriately during subsequent analysis.

### LBXTR — Triglyceride Level

- The mean triglyceride level is **120.86 mg/dL**, with a standard deviation of **124.67 mg/dL**.
- The median is **95 mg/dL**, while the maximum observed value is **4,233 mg/dL**.
- The difference between the mean and median, along with the high maximum value, suggests a right-skewed distribution.
- Approximately **55.75%** of the values are missing (3,216 out of 5,769 participants).
- The unusually high maximum value will not be removed automatically without further investigation.
- Mean or median imputation is not appropriate at this stage because of the substantial missingness.
- The variable is retained for further analysis using its available observations.

### Overall Decision

- `SMD650` is removed because approximately 79.56% of its values are missing.
- `SMQ040` is retained as a categorical variable despite its high missingness because it provides information about current smoking status.
- `LBDLDL` and `LBXTR` are retained because they provide important laboratory measurements relevant to the study.
- No automatic outlier removal or numerical imputation is performed for these laboratory variables at this stage.
- The remaining missing values will be considered during subsequent preprocessing and statistical analysis.

In [110]:
processed_data.drop(columns=["SMD650"], inplace=True)

## Analysis of Moderately Missing Value Columns

In [111]:
moderately_missing_cols = missing_stats[missing_stats["Category"] == "Moderate missingness"].index
display(missing_stats[missing_stats["Category"] == "Moderate missingness"])

,Missing_Values,Missing_Values_Proportion(%),Description,Category
ALQ120U,2321,40.232276,Unit associated with the alcohol consumption f...,Moderate missingness
ALQ120Q,1452,25.169007,Alcohol consumption frequency.,Moderate missingness


In [112]:
# Categorize alcohol frequency responses
q_status = pd.Series("Reported frequency",index=processed_data.index)

q_status.loc[processed_data["ALQ120Q"] == 0] = "Never (0)"
q_status.loc[processed_data["ALQ120Q"].isna()] = "Missing"

# Compare frequency responses with missing units
display(pd.crosstab(q_status,processed_data["ALQ120U"].isna(),margins=True))

# Check valid unit distribution among positive frequencies
print("\nAlcohol frequency units:")
display(processed_data.loc[processed_data["ALQ120Q"] > 0, "ALQ120U"].value_counts(dropna=False).sort_index())

ALQ120U,False,True,All
row_0,,,
Missing,0,1452,1452
Never (0),0,869,869
Reported frequency,3448,0,3448
All,3448,2321,5769



Alcohol frequency units:


ALQ120U
1.0    1434
2.0     905
3.0    1109
Name: count, dtype: int64

### ALQ120Q — Alcohol Consumption Frequency

- `ALQ120Q` represents the reported frequency of alcohol consumption.
- A total of **1,452 values (25.17%)** are missing among 5,769 participants.
- **869 participants** reported a value of `0`, indicating that they did not consume alcohol during the reference period.
- The remaining **3,448 participants** reported a positive alcohol consumption frequency.
- Missing values are retained as `NaN` because they represent unavailable responses and should not be confused with valid zero responses.
- The variable is retained because it provides useful information about participants' alcohol consumption patterns.

### ALQ120U — Alcohol Consumption Frequency Unit

- `ALQ120U` represents the unit associated with the reported alcohol consumption frequency.
- A total of **2,321 values (40.23%)** are missing.
- Further investigation revealed that all missing values occur among participants with either a missing `ALQ120Q` response or a valid zero response.
- Among the 3,448 participants reporting a positive alcohol consumption frequency, the unit is available for every participant.
- The observed unit distribution is:
  - Code `1`: 1,434 participants.
  - Code `2`: 905 participants.
  - Code `3`: 1,109 participants.
- Therefore, the missingness in `ALQ120U` is associated with the response pattern of `ALQ120Q`, rather than missing units among participants who reported a positive frequency.
- The variable is retained without imputation because assigning an artificial unit to zero or unavailable frequency responses would be inappropriate.

### Overall Decision

- Both `ALQ120Q` and `ALQ120U` are retained in the dataset.
- No mean, median, or mode imputation is performed.
- Valid zero responses in `ALQ120Q` are preserved.
- Missing responses in `ALQ120Q` remain missing.
- Missing values in `ALQ120U` are preserved because the unit is unnecessary when the frequency is zero or unavailable.

Overall, both variables provide complementary information about alcohol consumption. Their relationship must be considered during subsequent analysis to avoid treating structurally associated missing values as ordinary missing observations.

## Analysis of Low Missing Value Columns

In [113]:
missing_stats[missing_stats["Category"] == "Low missingness"]

,Missing_Values,Missing_Values_Proportion(%),Description,Category
DR1TTFAT,722,12.515167,Total fat intake.,Low missingness
DR1TSFAT,722,12.515167,Saturated fat intake.,Low missingness
DR1TCHOL,722,12.515167,Cholesterol intake.,Low missingness
DR1TKCAL,722,12.515167,Energy intake.,Low missingness
DR1TSUGR,722,12.515167,Total sugar intake.,Low missingness
DR1TFIBE,722,12.515167,Dietary fibre intake.,Low missingness
DR1TALCO,722,12.515167,Alcohol intake.,Low missingness
DR1TCAFF,722,12.515167,Caffeine intake.,Low missingness
DR1TCARB,722,12.515167,Carbohydrate intake.,Low missingness
DR1TPROT,722,12.515167,Protein intake.,Low missingness


### DR1TCHOL, DR1TKCAL, DR1TALCO, DR1TCAFF, DR1TCARB, DR1PROT, DR1TFIBE, DRTSUGR, DR1TSFAT, DR1TTFAT

In [114]:
missing_stats[missing_stats["Category"] == "Low missingness"].head(10)

,Missing_Values,Missing_Values_Proportion(%),Description,Category
DR1TTFAT,722,12.515167,Total fat intake.,Low missingness
DR1TSFAT,722,12.515167,Saturated fat intake.,Low missingness
DR1TCHOL,722,12.515167,Cholesterol intake.,Low missingness
DR1TKCAL,722,12.515167,Energy intake.,Low missingness
DR1TSUGR,722,12.515167,Total sugar intake.,Low missingness
DR1TFIBE,722,12.515167,Dietary fibre intake.,Low missingness
DR1TALCO,722,12.515167,Alcohol intake.,Low missingness
DR1TCAFF,722,12.515167,Caffeine intake.,Low missingness
DR1TCARB,722,12.515167,Carbohydrate intake.,Low missingness
DR1TPROT,722,12.515167,Protein intake.,Low missingness


In [115]:
diet_cols = [
    "DR1TTFAT", "DR1TSFAT", "DR1TCHOL",
    "DR1TKCAL", "DR1TSUGR", "DR1TFIBE",
    "DR1TALCO", "DR1TCAFF", "DR1TCARB",
    "DR1TPROT"
]

diet_missing = processed_data[diet_cols].isna()

print("Missing values per variable:")
print(diet_missing.sum())

print("\nParticipants missing all 10 dietary variables:",
      diet_missing.all(axis=1).sum())

print("\nParticipants missing at least one dietary variable:",
      diet_missing.any(axis=1).sum())

print("\nParticipants with mixed missingness:",
      (diet_missing.any(axis=1) & ~diet_missing.all(axis=1)).sum())

Missing values per variable:
DR1TTFAT    722
DR1TSFAT    722
DR1TCHOL    722
DR1TKCAL    722
DR1TSUGR    722
DR1TFIBE    722
DR1TALCO    722
DR1TCAFF    722
DR1TCARB    722
DR1TPROT    722
dtype: int64

Participants missing all 10 dietary variables: 722

Participants missing at least one dietary variable: 722

Participants with mixed missingness: 0


#### Dietary Variables

- All ten selected dietary variables (`DR1TTFAT`, `DR1TSFAT`, `DR1TCHOL`, `DR1TKCAL`, `DR1TSUGR`, `DR1TFIBE`, `DR1TALCO`, `DR1TCAFF`, `DR1TCARB`, and `DR1TPROT`) contain exactly 722 missing values each, representing approximately **12.52%** of the eligible participants.
- Further investigation revealed that all 722 participants have missing values across all ten dietary variables.
- The remaining 5,047 participants have complete observations for all ten dietary variables.
- No participants have partially missing dietary measurements.
- This indicates a common participant-level missingness pattern rather than independent missingness across individual dietary variables.
- All ten variables are retained because they provide useful information about dietary intake and nutritional characteristics.
- No individual mean or median imputation is performed at this stage.

Overall, the dietary variables exhibit consistent missingness across participants. Their missing values should be handled together when conducting subsequent statistical analyses.

### ALQ101 - History of alcohol consumption

- It is a categorial column, **1** for Yes, **2** for No
- Mean of 1.29 shows, most participants have consumed alcohol
- Missing values are retained as `NaN`.

In [116]:
print("ALQ101 distribution:")

alq101_counts = processed_data["ALQ101"].value_counts(
    dropna=False
)

print(alq101_counts.rename({
    1.0: "Yes",
    2.0: "No"
}))

print("\nMissing percentage:",processed_data["ALQ101"].isna().mean() * 100)

ALQ101 distribution:
ALQ101
Yes    3666
No     1428
NaN     675
Name: count, dtype: int64

Missing percentage: 11.700468018720748


In [117]:
# Another alcohol consumption related column
print(column_descriptions["ALQ120Q"])

Alcohol consumption frequency.


In [118]:
# Inspect the adults with missing ALQ101 but available ALQ120Q
processed_data[processed_data["ALQ101"].isna() &processed_data["ALQ120Q"].notna()][["SEQN", "RIDAGEYR", "ALQ101", "ALQ120Q"]]

,SEQN,RIDAGEYR,ALQ101,ALQ120Q
250,73807,50,NaN,6.0
916,74473,80,NaN,0.0
2883,76440,49,NaN,6.0
4462,78019,65,NaN,0.0
6765,80322,55,NaN,0.0
7970,81527,35,NaN,1.0


In [119]:
# Inspect the six unusual alcohol records
cols = ["SEQN", "RIDAGEYR", "ALQ101", "ALQ120Q"]

display(processed_data[processed_data["ALQ101"].isna() & processed_data["ALQ120Q"].notna()][cols])

,SEQN,RIDAGEYR,ALQ101,ALQ120Q
250,73807,50,NaN,6.0
916,74473,80,NaN,0.0
2883,76440,49,NaN,6.0
4462,78019,65,NaN,0.0
6765,80322,55,NaN,0.0
7970,81527,35,NaN,1.0


In [120]:
# Get the SEQN values of the six unusual records
ids = processed_data[processed_data["ALQ101"].isna() & processed_data["ALQ120Q"].notna()]["SEQN"]

# Check the original questionnaire dataset
display(questionnaire_cleaned[questionnaire_cleaned["SEQN"].isin(ids)][["SEQN", "ALQ101", "ALQ120Q"]])

,SEQN,ALQ101,ALQ120Q
250,73807,NaN,6.0
916,74473,NaN,0.0
2883,76440,NaN,6.0
4462,78019,NaN,0.0
6765,80322,NaN,0.0
7970,81527,NaN,1.0


The remaining missing values occur in eligible adults and are also present in the original questionnaire data. They are retained as `NaN` rather than being inferred from `ALQ120Q`.

### BPXSY1 - Systolic blood pressure.

- Values have high std deviation and the distribution is right skewed
- It is a clinical and important reading, hence missing values must be retained as NaN, to avoid artifical measurements

In [121]:
print(column_descriptions["BPXSY1"])
print(processed_data["BPXSY1"].describe())

Systolic blood pressure.
count    5111.000000
mean      123.161025
std        18.116186
min        66.000000
25%       110.000000
50%       120.000000
75%       134.000000
max       228.000000
Name: BPXSY1, dtype: float64


In [122]:
processed_data["BPXSY1"].isna().sum()

np.int64(658)

### BPXDI1 - Diastolic blood pressure.

- Similar to BPXSY1, we should retain missing values as NaN
- All the participants who have missing BPXDI1 also have missing BPXSY1

In [123]:
print(column_descriptions["BPXDI1"])
print(processed_data["BPXDI1"].describe())

Diastolic blood pressure.
count    5111.000000
mean       69.912346
std        12.809554
min         0.000000
25%        62.000000
50%        70.000000
75%        78.000000
max       122.000000
Name: BPXDI1, dtype: float64


In [124]:
processed_data["BPXDI1"].isna().sum()

np.int64(658)

Check whether the same participant have both the readings missing

In [125]:
print("Participants with both BPXSY1 and BPXDI1 missing:", processed_data[(processed_data["BPXSY1"].isna() == True) & (processed_data["BPXDI1"].isna() == True)].shape[0])

Participants with both BPXSY1 and BPXDI1 missing: 658


This shows this is clinical missingness and shouldn't we replaced

### Creating A Mean Systolic and Dystolic Blood Pressure Column

In [126]:
# Calculate mean systolic blood pressure using available readings
systolic_cols = ["BPXSY1", "BPXSY2", "BPXSY3"]

processed_data["SBP_mean"] = processed_data[
    systolic_cols
].mean(axis=1, skipna=True)

In [127]:
# Number of available readings per participant
sbp_reading_count = processed_data[systolic_cols].notna().sum(axis=1)

print("Available systolic readings:")
print(sbp_reading_count.value_counts().sort_index())

print("\nMean systolic blood pressure statistics:")
print(processed_data["SBP_mean"].describe())

print("\nParticipants with missing SBP_mean:",
      processed_data["SBP_mean"].isna().sum())

Available systolic readings:
0     381
1      32
2     384
3    4972
Name: count, dtype: int64

Mean systolic blood pressure statistics:
count    5388.000000
mean      123.381651
std        17.951347
min        64.666667
25%       110.666667
50%       120.000000
75%       133.333333
max       228.666667
Name: SBP_mean, dtype: float64

Participants with missing SBP_mean: 381


In [128]:
diastolic_cols = ["BPXDI1", "BPXDI2", "BPXDI3"]

processed_data["DBP_mean"] = processed_data[
    diastolic_cols
].mean(axis=1, skipna=True)

In [129]:
dbp_reading_count = processed_data[diastolic_cols].notna().sum(axis=1)

print("Available diastolic readings:")
print(dbp_reading_count.value_counts().sort_index())

print("\nMean diastolic blood pressure:")
print(processed_data["DBP_mean"].describe())

print("\nMissing DBP_mean:",
      processed_data["DBP_mean"].isna().sum())

Available diastolic readings:
0     381
1      32
2     384
3    4972
Name: count, dtype: int64

Mean diastolic blood pressure:
count    5388.000000
mean       69.604739
std        12.711620
min         0.000000
25%        63.333333
50%        70.666667
75%        77.000000
max       116.666667
Name: DBP_mean, dtype: float64

Missing DBP_mean: 381


In [130]:
bp_original_cols = [
    "BPXSY1", "BPXSY2", "BPXSY3",
    "BPXDI1", "BPXDI2", "BPXDI3"
]

processed_data.drop(columns=bp_original_cols, inplace=True)

In [131]:
print("Remaining blood pressure columns:")
print([col for col in processed_data.columns if col in ["SBP_mean", "DBP_mean"]])

Remaining blood pressure columns:
['SBP_mean', 'DBP_mean']


In [132]:
# Remove participants with no available blood pressure readings
before_rows = len(processed_data)

processed_data.dropna(subset=["SBP_mean", "DBP_mean"],how="all",inplace=True)

In [133]:
after_rows = len(processed_data)

print("Participants before removal:", before_rows)
print("Participants removed:", before_rows - after_rows)
print("Participants remaining:", after_rows)

print("\nMissing SBP_mean:", processed_data["SBP_mean"].isna().sum())
print("Missing DBP_mean:", processed_data["DBP_mean"].isna().sum())

Participants before removal: 5769
Participants removed: 381
Participants remaining: 5388

Missing SBP_mean: 0
Missing DBP_mean: 0


### BMXWAIST - Waist Circumference
- Out of 5,388 participants, 5,126 have available measurements and 262 are missing (4.86%).
- The mean waist circumference is 99.13 cm, with a standard deviation of 16.53 cm. The median is 97.50 cm.
- Missingness is slightly higher among females (5.32%) than males (4.36%).
- Missingness increases with age, reaching 19.03% among participants aged 80 and above.
- The variable is retained because it provides useful information about abdominal adiposity and metabolic health.
- The 262 missing values are retained as NaN. No mean or median imputation is performed.

Overall, waist circumference has relatively low missingness and is retained for subsequent statistical analysis using available observations.

In [134]:
print(column_descriptions["BMXWAIST"])
print(processed_data["BMXWAIST"].describe())

Waist circumference.
count    5126.000000
mean       99.133028
std        16.534476
min        55.500000
25%        87.300000
50%        97.500000
75%       108.500000
max       177.900000
Name: BMXWAIST, dtype: float64


In [135]:
processed_data["BMXWAIST"].isna().sum()

np.int64(262)

In [136]:
print("Missing waist measurements by sex:")
print(processed_data.groupby("RIAGENDR", observed=True)["BMXWAIST"].apply(lambda x: x.isna().sum()))

print("\nMissing waist measurements by age group:")
print(processed_data.groupby("AgeGroup", observed=True)["BMXWAIST"].apply(lambda x: x.isna().sum()))

Missing waist measurements by sex:
RIAGENDR
1    113
2    149
Name: BMXWAIST, dtype: int64

Missing waist measurements by age group:
AgeGroup
20-29    23
30-39    28
40-49    38
50-59    30
60-69    44
70-79    40
80+      59
Name: BMXWAIST, dtype: int64


In [137]:
# Missingness by sex
sex_missing = processed_data.groupby("RIAGENDR").agg(Total=("BMXWAIST", "size"), Missing=("BMXWAIST", lambda x: x.isna().sum()))

sex_missing["Missing_Percentage"] = (sex_missing["Missing"] / sex_missing["Total"] * 100)

print("Missingness by sex:")
display(sex_missing)


# Missingness by age group
age_missing = processed_data.groupby("AgeGroup", observed=True).agg(
    Total=("BMXWAIST", "size"),
    Missing=("BMXWAIST", lambda x: x.isna().sum())
)

age_missing["Missing_Percentage"] = (age_missing["Missing"] / age_missing["Total"] * 100)

print("Missingness by age group:")
display(age_missing)

Missingness by sex:


,Total,Missing,Missing_Percentage
RIAGENDR,,,
1,2589,113,4.36462
2,2799,149,5.32333


Missingness by age group:


,Total,Missing,Missing_Percentage
AgeGroup,,,
20-29,889,23,2.587177
30-39,935,28,2.994652
40-49,967,38,3.929679
50-59,885,30,3.389831
60-69,883,44,4.983012
70-79,519,40,7.707129
80+,310,59,19.032258


### INDFMPIR - Family income-to-poverty ratio.

- The data is right-skewed
- Used median to fill the missing values because

In [138]:
print(column_descriptions["INDFMPIR"])
print(processed_data["INDFMPIR"].describe())

Family income-to-poverty ratio.
count    4977.000000
mean        2.509612
std         1.648682
min         0.000000
25%         1.050000
50%         2.120000
75%         4.120000
max         5.000000
Name: INDFMPIR, dtype: float64


In [139]:
processed_data["INDFMPIR"].isna().sum()

np.int64(411)

In [140]:
i_to_p_median_val = processed_data["INDFMPIR"].median()
processed_data["INDFMPIR"] = processed_data["INDFMPIR"].fillna(i_to_p_median_val)

In [141]:
processed_data["INDFMPIR"].isna().sum()

np.int64(0)

### LBXSASSI - Aspartate transaminase
- The distribution is right-skewed, with a mean of 25.12, median of 22, and a maximum of 882.
- The high values represent potential extreme observations and should not be removed automatically.
- Missing values are retained as `NaN`

In [142]:
processed_data["LBXSASSI"].describe()

count    5145.000000
mean       25.497182
std        19.108861
min         9.000000
25%        19.000000
50%        22.000000
75%        27.000000
max       882.000000
Name: LBXSASSI, dtype: float64

In [143]:
column_descriptions["LBXSASSI"] 
# Aspartate transaminase

'AST.'

### LBXSATSI - Alanine Aminotransferase
- Maximum value is much larger than the typical values
- Missing values are retained as `NaN` because the laboratory missingness is substantial and appears to follow the laboratory sampling pattern.


In [144]:
column_descriptions["LBXSATSI"]
# Alanine Aminotransferase. its higher amount can be associated with liver injury or inflammation

'ALT.'

In [145]:
processed_data["LBXSATSI"].describe()

count    5145.000000
mean       25.065112
std        19.344242
min         6.000000
25%        16.000000
50%        20.000000
75%        28.000000
max       536.000000
Name: LBXSATSI, dtype: float64

In [146]:
processed_data["LBXSATSI"].isnull().sum()

np.int64(243)

### LBXSCH - Total Cholestrol
- 2,021 eligible participants had missing LBXSCH.
- Missing values are largely associated with missing laboratory measurements. They are retained as `NaN` rather than being replaced with mean or median values.


In [147]:
column_descriptions["LBXSCH"]

'Total cholesterol.'

In [148]:
processed_data["LBXSCH"].isna().sum()

np.int64(243)

In [149]:
print(column_descriptions["LBXSGL"])
print(column_descriptions["LBXGH"])

Blood glucose.
HbA1c.


In [150]:
# Check whether participants missing total cholesterol also have missing glucose and glycohemoglobin
chol_missing = processed_data[processed_data["LBXSCH"].isna()]

print("Missing LBXSCH (Cholestrol):", len(chol_missing))
print("Also missing LBXSGL (Glucose):", chol_missing["LBXSGL"].isna().sum())
print("Also missing LBXGH (HbA1c):", chol_missing["LBXGH"].isna().sum())

Missing LBXSCH (Cholestrol): 243
Also missing LBXSGL (Glucose): 241
Also missing LBXGH (HbA1c): 175


- For most participant, both the cholestrol, glucose and glycohemoglobin readings are missing
- This shows lab missingness

In [151]:
# Check how many other selected laboratory values are missing
lab_columns = ["LBXSGL", "LBXGH", "LBXSCH", "LBDHDD", "LBDLDL","LBXTR", "LBXSATSI", "LBXSASSI", "LBXSGTSI","LBXSCR", "LBXSUA", "LBDSALSI", "LBXLYPCT", "LBXNEPCT"]

chol_missing = processed_data[processed_data["LBXSCH"].isna()]
display(chol_missing[lab_columns].isnull().sum().sort_values(ascending=False))

LBXSCH      243
LBXSUA      243
LBXSASSI    242
LBXSGTSI    242
LBXSATSI    242
LBXSGL      241
LBXSCR      241
LBDSALSI    241
LBDLDL      233
LBXTR       233
LBDHDD      227
LBXGH       175
LBXLYPCT    174
LBXNEPCT    174
dtype: int64

Missing values are largely associated with missing laboratory measurements. 
They are kept as NaN rather been replaced with mean or median

### LBXSUA - Uric Acid

 - The observed values show slight right skewness. 
 - Missing values are retained as `NaN` because they are part of the broader laboratory missingness pattern.

In [152]:
column_descriptions["LBXSUA"]

'Uric acid.'

In [153]:
processed_data["LBXSUA"].describe()

count    5145.000000
mean        5.428999
std         1.423750
min         0.700000
25%         4.400000
50%         5.300000
75%         6.300000
max        13.300000
Name: LBXSUA, dtype: float64

In [154]:
processed_data["LBXSUA"].isnull().sum()

np.int64(243)

### LBXSGTSI
- Distribution is strongly right-skewed

In [155]:
column_descriptions["LBXSGTSI"] # Gamma-glutamyl transferase

'GGT.'

In [156]:
processed_data["LBXSGTSI"].describe()

count    5146.000000
mean       28.672756
std        47.153421
min         5.000000
25%        14.000000
50%        19.000000
75%        29.000000
max      1510.000000
Name: LBXSGTSI, dtype: float64

In [157]:
processed_data["RIDAGEYR"].describe()

count    5388.000000
mean       49.075167
std        17.481585
min        20.000000
25%        34.000000
50%        48.000000
75%        63.000000
max        80.000000
Name: RIDAGEYR, dtype: float64

### LBDSALSI - Albumin

-  The distribution is approximately symmetric, with a mean of 42.82 g/L and median of 43 g/L.
-  Missing values are retained as `NaN` because they are part of the broader laboratory missingness pattern.

In [158]:
print(column_descriptions["LBDSALSI"])
print(processed_data["LBDSALSI"].describe())

Albumin.
count    5147.000000
mean       42.462794
std         3.403538
min        24.000000
25%        40.000000
50%        43.000000
75%        45.000000
max        56.000000
Name: LBDSALSI, dtype: float64


In [159]:
processed_data["LBDSALSI"].isnull().sum()

np.int64(241)

### LBXSCR - Creatinine

- The distribution is right-skewed, with a mean of 0.88 mg/dL and median of 0.82 mg/dL.
- Missing values are retained as `NaN` because they are part of the broader laboratory missingness pattern and the amount of missing values are same as the other parameters.

In [160]:
print(column_descriptions["LBXSCR"])
print(processed_data["LBXSCR"].describe())

Creatinine.
count    5147.000000
mean        0.911010
std         0.447095
min         0.300000
25%         0.720000
50%         0.850000
75%         1.010000
max        16.640000
Name: LBXSCR, dtype: float64


In [161]:
processed_data["LBXSCR"].isnull().sum()

np.int64(241)

### LBXSGL - Blood Glucose 
- The distribution is right-skewed, with a mean of 102.29 mg/dL and median of 93 mg/dL. Missing values are retained as `NaN` because they are part of the broader laboratory missingness pattern as already analyzed during previous columns/parameters

In [162]:
print(column_descriptions["LBXSGL"])
print(processed_data["LBXSGL"].describe())

Blood glucose.
count    5147.000000
mean      104.998057
std        41.831078
min        49.000000
25%        87.000000
50%        95.000000
75%       106.000000
max       577.000000
Name: LBXSGL, dtype: float64


In [163]:
processed_data["LBXSGL"].isnull().sum()

np.int64(241)

### LBDHDD - HDL cholesterol

In [164]:
print(column_descriptions["LBDHDD"])
print(processed_data["LBDHDD"].describe())

HDL cholesterol.
count    5161.000000
mean       52.792482
std        16.102835
min        10.000000
25%        41.000000
50%        50.000000
75%        62.000000
max       173.000000
Name: LBDHDD, dtype: float64


In [165]:
processed_data["LBDHDD"].isna().sum()

np.int64(227)

### LBXNEPCT - Neutrophil percentage

In [166]:
print(column_descriptions["LBXNEPCT"])
print(processed_data["LBXNEPCT"].describe())

Neutrophil percentage.
count    5195.000000
mean       57.985929
std         9.303982
min         8.800000
25%        52.000000
50%        58.500000
75%        64.200000
max        90.800000
Name: LBXNEPCT, dtype: float64


In [167]:
processed_data["LBXNEPCT"].isna().sum()

np.int64(193)

In [168]:
print(column_descriptions["DR1TKCAL"])
print(processed_data["DR1TKCAL"].describe())

Energy intake.
count     4931.000000
mean      2115.468059
std       1034.087329
min        117.000000
25%       1428.000000
50%       1935.000000
75%       2581.000000
max      12108.000000
Name: DR1TKCAL, dtype: float64


In [169]:
print(column_descriptions["DR1TPROT"])
print(processed_data["DR1TPROT"].describe())

Protein intake.
count    4931.000000
mean       82.694230
std        46.178825
min         0.000000
25%        52.165000
50%        74.100000
75%       101.785000
max       557.870000
Name: DR1TPROT, dtype: float64


In [170]:
processed_data["DR1TPROT"].isna().sum()

np.int64(457)

In [171]:
print(column_descriptions["DR1TSUGR"])
print(processed_data["DR1TSUGR"].describe())

Total sugar intake.
count    4931.000000
mean      110.379152
std        80.089654
min         0.130000
25%        57.410000
50%        92.970000
75%       142.290000
max      1115.500000
Name: DR1TSUGR, dtype: float64


### LBXLYPCT - Lymphocyte percentage

- Distribution is slightly right-skewed.
- Being a clinical measurement, we should not replace missing values

In [172]:
print(column_descriptions["LBXLYPCT"])
print(processed_data["LBXLYPCT"].describe())

Lymphocyte percentage.
count    5195.000000
mean       30.348989
std         8.478920
min         3.400000
25%        24.600000
50%        29.900000
75%        35.600000
max        88.000000
Name: LBXLYPCT, dtype: float64


In [173]:
processed_data["LBXLYPCT"].isna().sum()

np.int64(193)

### LBXGH - HbA1c

- The distribution is right-skewed, with a mean of 5.64% and median of 5.40%. Missing values are retained as `NaN` because they are part of the broader laboratory missingness pattern.

In [174]:
print(column_descriptions["LBXGH"])
print(processed_data["LBXGH"].describe())

HbA1c.
count    5209.000000
mean        5.735362
std         1.071894
min         3.500000
25%         5.200000
50%         5.500000
75%         5.900000
max        17.500000
Name: LBXGH, dtype: float64


In [175]:
processed_data["LBXGH"].isnull().sum()

np.int64(179)

## Checking the columsn with similar missginess

In [176]:
lab_cols = [
    "LBXSUA", "LBXSATSI", "LBXSCH", "LBXSASSI",
    "LBXSGTSI", "LBXSGL", "LBDSALSI", "LBXSCR",
    "LBDHDD", "LBXLYPCT", "LBXNEPCT", "LBXGH"
]

# Create missingness matrix
lab_missing = processed_data[lab_cols].isna()

# Missing values per variable
print("Missing values per laboratory variable:")
print(lab_missing.sum())

# Participants with all 12 measurements missing
all_missing = lab_missing.all(axis=1).sum()

# Participants with at least one missing measurement
any_missing = lab_missing.any(axis=1).sum()

# Participants with partially missing measurements
partial_missing = any_missing - all_missing

# Participants with all 12 measurements available
complete = (~lab_missing.any(axis=1)).sum()

print("\nMissingness pattern:")
print("Missing all 12:", all_missing)
print("Missing at least one:", any_missing)
print("Partially missing:", partial_missing)
print("All 12 available:", complete)

# Check whether every variable has exactly the same missing rows
same_pattern = all(
    lab_missing[col].equals(lab_missing[lab_cols[0]])
    for col in lab_cols[1:]
)

print("\nIdentical missingness across all variables:", same_pattern)

Missing values per laboratory variable:
LBXSUA      243
LBXSATSI    243
LBXSCH      243
LBXSASSI    243
LBXSGTSI    242
LBXSGL      241
LBDSALSI    241
LBXSCR      241
LBDHDD      227
LBXLYPCT    193
LBXNEPCT    193
LBXGH       179
dtype: int64

Missingness pattern:
Missing all 12: 174
Missing at least one: 267
Partially missing: 93
All 12 available: 5121

Identical missingness across all variables: False


## Removing Participant With One Lab Missing

In [177]:
# Remove participants with at least one missing laboratory measurement
before_rows = len(processed_data)

processed_data.dropna(subset=lab_cols, inplace=True)

after_rows = len(processed_data)

print("Participants before removal:", before_rows)
print("Participants removed:", before_rows - after_rows)
print("Participants remaining:", after_rows)

print("\nRemaining missing laboratory values:")
print(processed_data[lab_cols].isnull().sum())

Participants before removal: 5388
Participants removed: 267
Participants remaining: 5121

Remaining missing laboratory values:
LBXSUA      0
LBXSATSI    0
LBXSCH      0
LBXSASSI    0
LBXSGTSI    0
LBXSGL      0
LBDSALSI    0
LBXSCR      0
LBDHDD      0
LBXLYPCT    0
LBXNEPCT    0
LBXGH       0
dtype: int64


### BPXPLS - Pulse rate

- The single participant with missing value is retained

In [178]:
print(column_descriptions["BPXPLS"])
print(processed_data["BPXPLS"].describe())

Pulse rate.
count    5120.000000
mean       72.562109
std        11.782312
min        40.000000
25%        64.000000
50%        72.000000
75%        80.000000
max       160.000000
Name: BPXPLS, dtype: float64


In [179]:
processed_data["BPXPLS"].isna().sum()

np.int64(1)

### BMXBMI - Body Mass Index

- As BMI varies strongly with age and sex, we have used Group Median strategy to fill the missing values

In [180]:
print(column_descriptions["BMXBMI"])
print(processed_data["BMXBMI"].describe())

Body Mass Index.
count    5073.000000
mean       29.078494
std         7.044906
min        14.100000
25%        24.200000
50%        27.800000
75%        32.400000
max        77.500000
Name: BMXBMI, dtype: float64


In [181]:
processed_data["BMXBMI"].isna().sum()

np.int64(48)

In [182]:
# Calculate median BMI for each sex and age group
bmi_medians = (processed_data.groupby(["RIAGENDR", "AgeGroup"], observed=True)["BMXBMI"].median())

# Fill missing BMI values group by group
for (sex, age_group), median_value in bmi_medians.items():
    mask = ((processed_data["RIAGENDR"] == sex) & (processed_data["AgeGroup"] == age_group) & (processed_data["BMXBMI"].isna()))
    processed_data.loc[mask, "BMXBMI"] = median_value

print("Missing BMXBMI after filling:",processed_data["BMXBMI"].isna().sum())

Missing BMXBMI after filling: 0


In [183]:
processed_data["BMXBMI"].isna().sum()

np.int64(0)

### BMXHT - Standing Height

- Missingness is low
- Replaced missing values with median as the distribution is roughly symmetric

In [184]:
print(column_descriptions["BMXHT"])
print(processed_data["BMXHT"].describe())

Standing height.
count    5080.000000
mean      167.185211
std        10.157872
min       136.300000
25%       159.800000
50%       166.900000
75%       174.300000
max       202.600000
Name: BMXHT, dtype: float64


In [185]:
processed_data["BMXHT"].isnull().sum()

np.int64(41)

In [186]:
# Restore original height values
original_height = final_dataset.set_index("SEQN")["BMXHT"]

processed_data["BMXHT"] = processed_data["SEQN"].map(original_height)

# Calculate median by sex and age group
height_group_median = (processed_data.groupby(["RIAGENDR", "AgeGroup"],observed=True)["BMXHT"].transform("median"))

# Fallback to sex-specific median if needed
height_sex_median = (processed_data.groupby("RIAGENDR")["BMXHT"].transform("median"))

processed_data["BMXHT"] = (processed_data["BMXHT"].fillna(height_group_median).fillna(height_sex_median))

# Final fallback
processed_data["BMXHT"] = (processed_data["BMXHT"].fillna(processed_data["BMXHT"].median()))

In [187]:
print("Missing values after imputation:",processed_data["BMXHT"].isna().sum())

Missing values after imputation: 0


### BMXWT - Body weight.

- Distribution is right-skewed
- But still due to low missingess, median imputation based on sex and age group is used

In [188]:
print(column_descriptions["BMXWT"])
print(processed_data["BMXWT"].describe())

Body weight.
count    5085.000000
mean       81.533845
std        21.884028
min        32.300000
25%        66.200000
50%        78.200000
75%        92.800000
max       222.600000
Name: BMXWT, dtype: float64


In [189]:
processed_data["BMXWT"].isna().sum()

np.int64(36)

In [190]:
# Restore original body weight measurements
original_weight = final_dataset.set_index("SEQN")["BMXWT"]

processed_data["BMXWT"] = processed_data["SEQN"].map(original_weight)

# Calculate median by sex and age group
weight_group_median = (processed_data.groupby(["RIAGENDR", "AgeGroup"],observed=True)["BMXWT"].transform("median"))

# Fallback to sex-specific median
weight_sex_median = (processed_data.groupby("RIAGENDR")["BMXWT"].transform("median"))

# Perform imputation
processed_data["BMXWT"] = (processed_data["BMXWT"].fillna(weight_group_median).fillna(weight_sex_median).fillna(processed_data["BMXWT"].median()))

In [191]:
print("Missing values after imputation:",
      processed_data["BMXWT"].isna().sum())

Missing values after imputation: 0


### SMQ020 - Ever smoked 100 or more cigarettes.
- Only 2 missing values
- Filled missing values with mode

In [192]:
print(column_descriptions["SMQ020"])
print(processed_data["SMQ020"].describe())

Ever smoked 100 or more cigarettes.
count    5119.000000
mean        1.563977
std         0.495938
min         1.000000
25%         1.000000
50%         2.000000
75%         2.000000
max         2.000000
Name: SMQ020, dtype: float64


In [193]:
processed_data["SMQ020"].value_counts(dropna=False)

SMQ020
2.0    2887
1.0    2232
NaN       2
Name: count, dtype: int64

As their are only 2 missing values, we can replace them with mode

In [194]:
mode_value = processed_data["SMQ020"].mode().iloc[0]
processed_data["SMQ020"] = processed_data["SMQ020"].fillna(mode_value)

In [195]:
processed_data["SMQ020"].isna().sum()

np.int64(0)

### PAQ620 - moderate-intensity physical activity at work.

- 1 = Yes, 2 = No
- Used mode imputation to fill missing values

In [196]:
print(column_descriptions["PAQ620"])
print(processed_data["PAQ620"].describe())

Moderate work-related activity.
count    5118.000000
mean        1.671942
std         0.469552
min         1.000000
25%         1.000000
50%         2.000000
75%         2.000000
max         2.000000
Name: PAQ620, dtype: float64


In [197]:
processed_data["PAQ620"].isnull().sum()

np.int64(3)

In [198]:
processed_data["PAQ620"].value_counts(dropna=False)

PAQ620
2.0    3439
1.0    1679
NaN       3
Name: count, dtype: int64

Their are only 3 participants will missing value

In [199]:
paq620_mode = processed_data["PAQ620"].mode().iloc[0]
processed_data["PAQ620"] = processed_data["PAQ620"].fillna(paq620_mode)

In [200]:
processed_data["PAQ620"].isna().sum()

np.int64(0)

# Final Preview and Exporting the Dataset

In [201]:
processed_data.drop(columns="AgeGroup", inplace=True)
del column_descriptions["AgeGroup"]

In [202]:
# Overall dataset summary
summary = pd.DataFrame({
    "Metric": [
        "Number of rows",
        "Number of columns",
        "Total missing values",
        "Columns with missing values"
    ],
    "final_dataset": [
        final_dataset.shape[0],
        final_dataset.shape[1],
        final_dataset.isna().sum().sum(),
        (final_dataset.isna().sum() > 0).sum()
    ],
    "processed_data": [
        processed_data.shape[0],
        processed_data.shape[1],
        processed_data.isna().sum().sum(),
        (processed_data.isna().sum() > 0).sum()
    ]
})

display(summary)

,Metric,final_dataset,processed_data
0,Number of rows,10175,5121
1,Number of columns,62,57
2,Total missing values,144728,15879
3,Columns with missing values,47,19


In [203]:
processed_data.to_csv("processed_dataset.csv", index=False)

In [204]:
graph_names.update({
    "SBP_mean": "Mean Systolic Blood Pressure",
    "DBP_mean": "Mean Diastolic Blood Pressure"
})

column_descriptions.update({
    "SBP_mean": "Mean of available systolic blood pressure readings (mmHg).",
    "DBP_mean": "Mean of available diastolic blood pressure readings (mmHg)."
})

In [205]:
graph_column_names = {}

for col in processed_data.columns.to_list():
    # Calculating the missing value proportion
    missing_values = int(processed_data[col].isna().sum())
    missing_values_prop = (missing_values/len(processed_data))*100
    
    graph_column_names[col] = {
        "name": graph_names[col],
        "description": column_descriptions[col],
        "missing_values": missing_values,
        "missing_proportion": missing_values_prop,
    }

In [206]:
import json
with open("graph_columns.json", "w") as file:
    json.dump(graph_column_names, file, indent=4)

In [207]:
processed_data

,SEQN,RIAGENDR,RIDAGEYR,RIDRETH1,INDFMPIR,MCQ220,SMQ020,SMQ040,ALQ101,ALQ120Q,...,Antidiabetic_Use,Thyroid_Medication_Use,Respiratory_Medication_Use,Antidepressant_Use,Anticoagulant_Antiplatelet_Use,GI_Medication_Use,Analgesic_Use,Antibiotic_Use,SBP_mean,DBP_mean
0,73557,1,69,4,0.84,2.0,1.0,3.0,1.0,1.0,...,2,2,2,2,2,2,2,2,112.666667,74.000000
1,73558,1,54,3,1.78,2.0,1.0,2.0,1.0,7.0,...,1,0,0,0,0,0,0,0,157.333333,61.333333
2,73559,1,72,3,4.51,1.0,1.0,3.0,1.0,0.0,...,1,0,0,0,0,0,0,0,142.000000,82.000000
4,73561,2,73,3,5.00,2.0,2.0,NaN,1.0,0.0,...,2,1,2,2,2,2,2,2,137.333333,86.666667
5,73562,1,56,1,4.79,2.0,1.0,3.0,1.0,5.0,...,0,0,0,0,0,0,0,0,157.333333,82.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10164,83721,1,52,3,5.00,2.0,2.0,NaN,1.0,3.0,...,0,0,0,0,0,0,0,0,109.333333,72.666667
10166,83723,1,61,1,3.53,2.0,2.0,NaN,1.0,3.0,...,0,0,0,0,0,0,0,0,140.000000,64.000000
10167,83724,1,80,3,3.09,1.0,1.0,3.0,1.0,0.0,...,0,0,0,0,0,0,0,0,166.666667,67.333333
10170,83727,1,26,2,2.12,2.0,2.0,NaN,1.0,1.0,...,0,0,0,0,0,0,0,0,112.666667,72.666667
